# Reproducing the analyses in *Self-reported side effects of botulinum toxin type A in medical and cosmetic use on Reddit*

This notebook recomputes every number, table and figure in the manuscript from the study's private
inputs, and checks each result against the value printed in the paper. It is written to be read from
top to bottom. Each analysis starts with a markdown cell that says what is computed and why that
method was chosen, followed by the code that computes it.

**What the notebook does**

1. Rebuilds the corpus counts and the analysis set from the filtered Reddit archive (Section 1).
2. Reads the stored model extractions and rebuilds the nested model-selection samples (Section 2).
3. Re-analyses the blinded human verification (Section 3).
4. Describes the full-corpus extraction (Section 4).
5. Compares the extraction configurations with one another (Section 5).
6. Runs the primary medical-versus-cosmetic comparison (Section 6).
7. Runs every sensitivity and robustness analysis (Section 7).
8. Redraws the manuscript figures (Section 8).
9. Prints a pass or fail line for every checked number (Section 9).

**What it does not do.** It does not call the language models again. Re-running the extraction would
cost money, would not be bit-for-bit reproducible because the providers' models are not deterministic,
and would send post text to an external service, which the approved research plan (Vísindasiðanefnd,
VSN2608019) allows only for the study's own extraction runs. The notebook therefore starts from the
stored extraction checkpoint. It also does not collect new annotations; it reads the exported
decisions of the three annotators.

**Data protection.** The inputs contain Reddit post text and usernames. They stay on the research
machine, and the notebook never prints them: every output below is a count, a rate, a test statistic,
a candidate term label or a community name. Reddit user-profile pages, whose names are usernames, are
never shown individually. The notebook refuses to run if the private inputs are not git-ignored.

**How to run.** From the repository root: `uv run jupyter lab data_analysis/12_reproduce_paper.ipynb`,
or non-interactively with
`uv run jupyter nbconvert --to notebook --execute --inplace data_analysis/12_reproduce_paper.ipynb`.
A full run takes about a minute on a laptop, most of it in the permutation tests.

**Relation to the numbered scripts.** The scripts `01` to `11` in `data_analysis/` produced the
archived outputs used while writing the paper. This notebook is an independent re-implementation of
the same analyses, written for reading. Seeds, record order and algorithms follow the scripts exactly,
so the permutation and bootstrap results agree to the last digit.

## 0. Setup

**What.** Imports, file locations, the fixed random seeds and the analysis constants.

**Why.** All configuration lives in one cell so that a reader can see every choice that affects a
result. The three seeds are those used in the study: 20260713 fixed the random order behind the
nested model-selection samples, 20260715 drives every permutation test, and 20260927 drives the
bootstrap and random-draw analyses added after the internal review. The 20-submission threshold is the
minimum pooled count for a candidate term to enter the primary profile test. The small-cell threshold
of 5 is applied to any table of counts that could be published, as the approved plan requires.

In [ ]:
# Standard library
import csv
import hashlib
import json
import math
import random
import re
import sqlite3
import subprocess
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

# Third-party libraries (versions are pinned in uv.lock)
import numpy as np
import pandas as pd
from scipy import sparse, stats
from statsmodels.stats.multitest import multipletests

# Locate the repository root, whether the notebook is started from the root or from data_analysis/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ANALYSIS = ROOT / "data_analysis"
PRIVATE = ANALYSIS / "02_llm_extraction_results"  # extraction checkpoint and annotation exports
CORPUS = ROOT / "filtered_dataset"  # the filtered Reddit archive
OUT = ANALYSIS / "12_notebook_outputs"  # aggregate outputs written by this notebook
OUT.mkdir(exist_ok=True)

# Random seeds used in the study
SAMPLING_SEED = 20260713  # random order behind the nested 10% and 1% samples
PERMUTATION_SEED = 20260715  # every label-permutation test
RESAMPLING_SEED = 20260927  # author bootstrap, random author draws, misclassification bootstrap

# Analysis constants
PERMUTATIONS = 10_000  # permutations for the primary and whole-corpus tests
STRATUM_PERMUTATIONS = 2_000  # permutations for single communities, periods and injection sites
MIN_COUNT = 20  # minimum pooled submissions for a term in the primary profile test
MIN_COUNT_SMALL = 10  # the same threshold for the small within-site strata
SMALL_CELL = 5  # counts below this are suppressed in any displayed count table
Z = 1.959963984540054  # standard normal quantile for 95% intervals

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print(f"repository: {ROOT}")

**What.** A guard that stops the notebook if the private inputs could be committed to git.

**Why.** The checkpoint and the corpus hold post text and usernames. The approved research plan
restricts them to the named researchers, so the notebook checks that git ignores them before it opens
anything.

In [ ]:
# Refuse to continue if any private input is visible to git
for path in (
    PRIVATE / "private_extraction.sqlite3",
    PRIVATE / "annotations_export.csv",
    PRIVATE / "model_outputs_export.csv",
    CORPUS / "submissions.jsonl",
):
    assert path.exists(), f"missing input: {path}"
    ignored = subprocess.run(["git", "check-ignore", "-q", str(path)], cwd=ROOT).returncode == 0
    assert ignored, f"{path} is not git-ignored; stop before it is committed"
print("private inputs present and git-ignored")

### Shared statistical helpers

**What.** Small functions used throughout: label normalisation, the Wilson interval for a proportion,
the Newcombe-Wilson interval for a difference of proportions, the Katz interval for a risk ratio, the
base-2 Jensen-Shannon divergence, Cohen's kappa, and the binary Fleiss' kappa and pairwise agreement
for three raters.

**Why these methods.**
* *Wilson intervals* behave well for proportions near 0 or 1, which occur throughout (accuracies near
  99%, rare terms), where the textbook Wald interval fails.
* *Newcombe-Wilson intervals* combine two Wilson intervals and keep good coverage for differences of
  proportions, including differences near a boundary.
* *Jensen-Shannon divergence* measures how different two term distributions are. Base 2 bounds it
  between 0 (identical) and 1 (no overlap), it is symmetric, and it stays finite when a term occurs in
  only one group, which a Kullback-Leibler divergence would not.
* *Cohen's kappa* corrects agreement between two extraction runs for chance; *Fleiss' kappa* does the
  same for three annotators.

In [ ]:
def norm(value):
    """Lower-case and collapse whitespace: the normalisation every script applies to labels."""
    return " ".join(str(value).casefold().split())


def wilson(k, n, z=Z):
    """Wilson score 95% interval for the proportion k / n."""
    if n <= 0:
        return (math.nan, math.nan)
    p = k / n
    denominator = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denominator
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return max(0.0, centre - half), min(1.0, centre + half)


def newcombe(k1, n1, k2, n2):
    """Newcombe-Wilson 95% interval for the difference p1 - p2 of two independent proportions."""
    p1, p2 = k1 / n1, k2 / n2
    low1, high1 = wilson(k1, n1)
    low2, high2 = wilson(k2, n2)
    difference = p1 - p2
    return (
        difference - math.sqrt((p1 - low1) ** 2 + (high2 - p2) ** 2),
        difference + math.sqrt((high1 - p1) ** 2 + (p2 - low2) ** 2),
    )


def katz_risk_ratio(k1, n1, k2, n2):
    """Risk ratio (k1/n1) / (k2/n2) with a Katz log interval; 0.5 is added to cells if a count is zero."""
    if k1 == 0 or k2 == 0:
        k1, k2, n1, n2 = k1 + 0.5, k2 + 0.5, n1 + 1.0, n2 + 1.0
    ratio = (k1 / n1) / (k2 / n2)
    se = math.sqrt(1 / k1 - 1 / n1 + 1 / k2 - 1 / n2)
    return ratio, math.exp(math.log(ratio) - Z * se), math.exp(math.log(ratio) + Z * se)


def jsd(left, right):
    """Base-2 Jensen-Shannon divergence between count vectors; each row of left is paired with right."""
    left = np.atleast_2d(np.asarray(left, dtype=float))
    right = np.atleast_2d(np.asarray(right, dtype=float))
    if np.any(left.sum(axis=1) == 0) or np.any(right.sum(axis=1) == 0):
        raise ValueError("each profile must contain at least one candidate term")
    p = left / left.sum(axis=1, keepdims=True)  # turn counts into probability distributions
    q = right / right.sum(axis=1, keepdims=True)
    m = (p + q) / 2  # the mixture both are compared with
    with np.errstate(divide="ignore", invalid="ignore"):
        kl_p = np.where(p > 0, p * np.log2(p / m), 0.0).sum(axis=1)
        kl_q = np.where(q > 0, q * np.log2(q / m), 0.0).sum(axis=1)
    return 0.5 * (kl_p + kl_q)


def cohen_kappa(a, b):
    """Cohen's kappa between two equally long sequences of labels."""
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b, strict=True)) / n
    count_a, count_b = Counter(a), Counter(b)
    expected = sum((count_a[v] / n) * (count_b[v] / n) for v in count_a.keys() | count_b.keys())
    if math.isclose(expected, 1.0):
        return 1.0 if math.isclose(observed, 1.0) else math.nan
    return (observed - expected) / (1 - expected)


def fleiss_binary(correct_counts, raters):
    """Fleiss' kappa for a binary rating, from the number of raters who marked each unit correct."""
    units = len(correct_counts)
    p_correct = sum(correct_counts) / (units * raters)
    expected = p_correct**2 + (1 - p_correct) ** 2
    observed = sum(c * c + (raters - c) ** 2 - raters for c in correct_counts) / (
        units * raters * (raters - 1)
    )
    return (observed - expected) / (1 - expected)


def pairwise_binary(correct_counts, raters):
    """Share of agreeing rater pairs, pooled over units."""
    pairs = raters * (raters - 1) / 2
    agreeing = sum((c * (c - 1) + (raters - c) * (raters - c - 1)) / 2 for c in correct_counts)
    return agreeing / (pairs * len(correct_counts))

### Checking results against the paper

**What.** Every reproduced number is passed to `check`, which compares it with the value printed in
the manuscript and records the outcome for the report in Section 9.

**Why.** A reproduction is only useful if it says where it agrees. A value passes when it rounds to the
printed value at the precision the paper uses, which is the strongest agreement a printed number
allows.

In [ ]:
CHECKS = []


def check(label, value, reported, decimals=0, scale=1.0):
    """Record whether value * scale rounds to the number printed in the paper."""
    shown = value * scale
    ok = abs(shown - reported) <= 0.5 * 10 ** (-decimals) + 1e-9
    CHECKS.append(
        {
            "quantity": label,
            "paper": reported,
            "recomputed": round(shown, decimals + 2),
            "result": "PASS" if ok else "FAIL",
        }
    )
    print(
        f"{'PASS' if ok else 'FAIL'}  {label}: recomputed {shown:,.{decimals + 2}f}, paper {reported:,}"
    )


def suppress_small(frame, columns, threshold=SMALL_CELL):
    """Replace counts below the small-cell threshold with '<5' before a table is displayed."""
    shown = frame.copy()
    for column in columns:
        shown[column] = shown[column].apply(lambda v: f"<{threshold}" if 0 < v < threshold else v)
    return shown

## 1. Corpus construction

**What.** The corpus is every submission in the Academic Torrents release of the Pushshift and Arctic
Shift monthly dumps from January 2015 to December 2025 whose title or body names a BoNT-A product
(brand names, international nonproprietary names, or *botulinum*, *botulinum toxin* and *botulinum toxin
type A*), matched on word boundaries. The collection code scanned all 132 monthly files in one
streaming pass (3.32 TiB compressed); those two figures come from the collection run and are not
re-derived here. This section recomputes the rest of the corpus table from the filtered files.

**Why.** Scanning the whole archive, rather than a list of communities chosen in advance, avoids
selecting the sampling frame on what the investigators expected to find; it is how r/noburp entered the
study. Matching only the title and body keeps a permalink or community name from making a record match.

In [ ]:
# The collection manifest records how many matched submissions carried text, how many were dropped for
# having no body, and how many were kept only as context for comments.
manifest = json.loads((CORPUS / "manifest.json").read_text())
matched = (
    manifest["submissions_kept"]
    + manifest["submissions_dropped_no_body"]
    + manifest["context_submissions"]
)
check("Matched submissions", matched, 171_895)
check("Matched submissions carrying text", manifest["submissions_kept"], 121_019)

**What.** One streaming pass over the 121,019 submissions with text. It counts the records matched by
the token *bonta*, the link and image posts that nonetheless carry a body, the communities and named
authors, and the median body length. It also stores, for each record, only the fields later analyses
need: author, community and creation time.

**Why these rules.**
* *Text rule.* A submission enters if it has a non-empty body that is not `[removed]` or `[deleted]`.
  Reddit's `is_self` flag is deliberately not used: 19,203 link or image posts carry a body, and
  selecting on the flag would have discarded all of them.
* *The* bonta *exclusion.* *Bonta* was added as a punctuation-free spelling of BoNT-A but is also a
  surname and a place name in video games. Submissions matched by *bonta* alone discuss none of the
  drug's uses and are excluded before extraction; the one that also names another term stays.
* *Named authors.* Accounts shown as `[deleted]` or `[removed]`, and Reddit's AutoModerator, cannot be
  linked to a person, so they are not counted as authors.

In [ ]:
UNLINKABLE = {"[deleted]", "[removed]", "AutoModerator"}


def record_id(record):
    """Reddit fullname of a submission, for example t3_abc123."""
    rid = str(record.get("name") or record.get("id") or "").strip()
    return rid if rid.startswith("t3_") else f"t3_{rid}"


bonta_any = bonta_only = 0
link_posts_with_body = []  # body lengths of link or image posts that carry a body
body_lengths = []  # body lengths of every submission with text
communities, authors = set(), set()
eligible_ids = []  # the analysis set, in file order
meta = {}  # record id -> (author or None, community, created_utc)

with (CORPUS / "submissions.jsonl").open(encoding="utf-8") as handle:
    for line in handle:
        record = json.loads(line)
        rid = record_id(record)
        terms = {str(t).strip().casefold() for t in record.get("_matched_terms") or []}
        body = record.get("selftext") or ""
        author = record.get("author")
        author = None if not author or author in UNLINKABLE else author
        community = str(record.get("subreddit") or "").casefold()
        # Corpus-table counts over every submission with text
        bonta_any += "bonta" in terms
        body_lengths.append(len(body))
        communities.add(community)
        if author is not None:
            authors.add(author)
        if not record.get("is_self"):
            link_posts_with_body.append(len(body))
        # The analysis set excludes records matched by "bonta" alone
        if terms == {"bonta"}:
            bonta_only += 1
            continue
        eligible_ids.append(rid)
        try:
            created = float(record.get("created_utc"))
        except (TypeError, ValueError):
            created = None
        meta[rid] = (author, community, created)

assert len(eligible_ids) == len(set(eligible_ids)), "record identifiers must be unique"
check("Submissions selected by bonta", bonta_any, 2_258)
check("Submissions selected by bonta alone (excluded)", bonta_only, 2_257)
check("Submissions eligible for extraction", len(eligible_ids), 118_762)
check("Link or image posts that carry a body", len(link_posts_with_body), 19_203)
check("Median body length of those posts (characters)", float(np.median(link_posts_with_body)), 415)
check("Communities with at least one matching submission with text", len(communities), 13_513)
check("Distinct named authors across submissions with text", len(authors), 80_700)
check("Median body length (characters)", float(np.median(body_lengths)), 640)

## 2. The extraction checkpoint and the model-selection samples

**What.** Every eligible submission was sent, as title and body only, to a language model with a
system prompt holding the complete JSON schema, and one schema-validated object came back per
submission. Four configurations were run: DeepSeek V4 Flash without reasoning on the full corpus
(`stage1`), DeepSeek V4 Flash with high reasoning on a nested 10% sample (`stage2`), GPT-5.6 Luna with
low reasoning on a nested 1% sample through an interactive API (`stage3`), and the same GPT-5.6 Luna
configuration on the full corpus through OpenAI's Batch API (`stage4`, the primary extraction). The
results are stored in a read-only SQLite checkpoint.

**Why.** The nested samples let the smaller runs be compared record by record with the larger ones.
The corpus was shuffled once with seed 20260713; the 10% sample is every tenth record of that order and
the 1% sample every hundredth, so the 1% sample lies inside the 10% sample. The first cell below
rebuilds both samples and confirms they are the samples the checkpoint was run on.

In [ ]:
con = sqlite3.connect(f"file:{PRIVATE / 'private_extraction.sqlite3'}?mode=ro", uri=True)

# Rebuild the seeded order and the nested samples exactly as the extraction runner did
order = list(eligible_ids)
random.Random(SAMPLING_SEED).shuffle(order)
sample_10 = set(order[9::10])  # the last record in each random block of ten
sample_1 = set(order[99::100])  # the last record in each random block of one hundred


def attempted(stage):
    """Record ids a configuration was run on: its successes plus its recorded failures."""
    done = {
        r for (r,) in con.execute("SELECT record_id FROM stage_results WHERE stage = ?", (stage,))
    }
    failed = {
        r for (r,) in con.execute("SELECT record_id FROM stage_failures WHERE stage = ?", (stage,))
    }
    return done | failed


assert attempted("stage2") == sample_10, "10% sample differs from the one extracted"
assert attempted("stage3") == sample_1, "1% sample differs from the one extracted"
assert sample_1 <= sample_10
check("Nested 10% sample size", len(sample_10), 11_876)
check("Nested 1% sample size", len(sample_1), 1_187)

**What.** Successes and failures per configuration, and the reason for the 15 failures of the primary
run.

**Why.** Failures are reported rather than silently dropped. The primary run's failures are summarised
by their error message only; no record identifier or text is shown.

In [ ]:
rows = []
for stage in ("stage1", "stage2", "stage3", "stage4"):
    successes = con.execute(
        "SELECT COUNT(*) FROM stage_results WHERE stage = ?", (stage,)
    ).fetchone()[0]
    failures = con.execute(
        "SELECT COUNT(*) FROM stage_failures WHERE stage = ?", (stage,)
    ).fetchone()[0]
    rows.append({"configuration": stage, "successes": successes, "failures": failures})
display(pd.DataFrame(rows))

# Group the primary run's failures by the first part of the provider's error message
reasons = Counter(
    (status, (error or "")[:95])
    for status, error in con.execute(
        "SELECT status_code, error FROM stage_failures WHERE stage = 'stage4'"
    )
)
for (status, message), n in reasons.items():
    print(f"{n} failures, HTTP {status}: {message}...")
attempts = Counter(
    n
    for (n,) in con.execute(
        "SELECT COUNT(*) FROM stage_failure_history WHERE stage = 'stage4' GROUP BY record_id"
    )
)
print("failure attempts per record (attempts: records):", dict(sorted(attempts.items())))

check("Primary extractions returned", rows[3]["successes"], 118_747)
check("Primary extractions rejected by the provider", rows[3]["failures"], 15)

**What.** Every stored extraction is reduced to the normalised fields the analyses use. Candidate
Preferred Terms are lower-cased and whitespace-normalised, products become a set, and the free-text
treatment area is replaced by a hash so that two runs can still be compared for equality without the
text being kept in memory.

**Why.** Verbatim side-effect spans and the treatment-area text are the fields through which a source
post could be found again. No analysis needs them, so they are dropped at load time. The field list and
normalisation follow the analysis scripts exactly, which matters for the full-record agreement in
Section 5.

In [ ]:
EXTRACTED_FIELDS = (
    "a_self_use",
    "b_products_mentioned",
    "c_indication_category",
    "c_indication_subtype",
    "d_side_effect_pts",
    "e1_post_overall_sentiment",
    "e2_sentiment_about_botox",
    "f_sentiment_about_side_effect",
    "g_treatment_area_freetext",
    "h_dose_units",
    "i_provider_type",
    "j_first_time_or_repeat",
    "j_prior_treatment_count",
    "k_time_to_onset",
    "l_duration_status",
    "m_sought_medical_attention",
    "n_discontinuation_intent",
    "o_age_range",
    "o_sex",
)


def analysis_view(extraction):
    """Normalised, text-free view of one extraction."""
    view = {}
    for field in EXTRACTED_FIELDS:
        value = extraction.get(field)
        if field == "b_products_mentioned":
            view[field] = frozenset(norm(item) for item in value or [])
        elif field == "d_side_effect_pts":
            # keep only the candidate term of each side effect, never its verbatim span
            view[field] = frozenset(
                norm(item.get("meddra_pt"))
                for item in value or []
                if isinstance(item, dict) and item.get("meddra_pt")
            )
        elif field == "g_treatment_area_freetext":
            text = norm(value) if value is not None else ""
            view[field] = hashlib.sha256(text.encode()).hexdigest() if text else None
        elif isinstance(value, str):
            view[field] = norm(value)
        else:
            view[field] = value
    view["side_effect_presence"] = bool(view["d_side_effect_pts"])
    return view


def load_views(stage):
    """All extractions of one configuration, keyed by record id, in checkpoint order."""
    return {
        rid: analysis_view(json.loads(payload))
        for rid, payload in con.execute(
            "SELECT record_id, extraction_json FROM stage_results WHERE stage = ?", (stage,)
        )
    }


views = {stage: load_views(stage) for stage in ("stage1", "stage2", "stage3", "stage4")}
print({stage: len(v) for stage, v in views.items()})

## 3. Model selection by human verification

**What.** Before the primary full-corpus run, two configurations were checked by hand: DeepSeek V4
Flash without reasoning and GPT-5.6 Luna with low reasoning, on the 1,168 submissions of the 1% sample
where both returned a result. Three annotators judged every field of both outputs, and each extracted
side-effect span and candidate term, as correct or incorrect against the source post, in an interface
that hid which model produced which output. This section recomputes accuracy, agreement between
annotators, the paired comparison of the two models, accuracy on informative values, and accuracy by
the indication group the model assigned.

**Why.** Model choice decides every downstream count, so it was made on human judgement rather than on
agreement between models. A field is counted correct when at least two of three annotators marked it
correct: a majority vote is the simplest rule that settles every disagreement among three raters. The
decisions are loaded without the annotators' free-text notes.

In [ ]:
# Load the annotation export without its free-text notes column
annotations = pd.read_csv(
    PRIVATE / "annotations_export.csv",
    usecols=["annotator", "record_id", "model_key", "model_output_id", "decisions_json"],
)

# Keep only annotators who completed every form; the other account was used for testing the interface
forms_per_annotator = annotations.groupby("annotator")["model_output_id"].nunique()
complete = sorted(forms_per_annotator[forms_per_annotator == forms_per_annotator.max()].index)
annotations = annotations[annotations["annotator"].isin(complete)]
RATERS = len(complete)
print(f"{RATERS} complete annotators; {forms_per_annotator.max():,} forms each")

SPAN = re.compile(r"^d_side_effect_pts\[(\d+)\]\.(raw_span|meddra_pt)$")


def canonical(key):
    """Map an indexed span key such as d_side_effect_pts[2].raw_span to its field name."""
    match = SPAN.fullmatch(key)
    return f"d_side_effect_pts[*].{match.group(2)}" if match else key


ratings = defaultdict(dict)  # (model output id, key) -> {annotator: judged correct}
unit_model = {}  # (model output id, key) -> model
unit_record = {}  # (model output id, key) -> source record id (never printed)
output_model = {}  # model output id -> model, in the order outputs first appear
for row in annotations.itertuples(index=False):
    for key, decision in json.loads(row.decisions_json).items():
        unit = (int(row.model_output_id), key)
        ratings[unit][row.annotator] = decision == "correct"
        unit_model[unit] = row.model_key
        unit_record[unit] = row.record_id
        output_model.setdefault(int(row.model_output_id), row.model_key)

assert all(len(r) == RATERS for r in ratings.values()), (
    "every unit needs a rating from every annotator"
)
majority = {unit: sum(r.values()) * 2 > RATERS for unit, r in ratings.items()}
correct_counts = {unit: sum(r.values()) for unit, r in ratings.items()}
check("Rating units", len(ratings), 45_444)
check("Field decisions", len(ratings) * RATERS, 136_332)

**What.** Agreement among the three annotators, pooled over all rating units: the share of units on
which all three agreed, the share of agreeing rater pairs, Fleiss' kappa, and the prevalence-adjusted
bias-adjusted kappa (PABAK).

**Why both kappas.** When nearly every rating says "correct", chance agreement is already high, so
Fleiss' kappa is low even when raters rarely disagree. PABAK, which for two categories equals twice the
observed agreement minus one, shows agreement without that prevalence effect. Reporting both lets the
reader see both views.

In [ ]:
counts = list(correct_counts.values())
unanimous = sum(c in (0, RATERS) for c in counts) / len(counts)
pairwise = pairwise_binary(counts, RATERS)
kappa = fleiss_binary(counts, RATERS)
pabak = 2 * pairwise - 1
check("Unanimous rating units (%)", unanimous, 94.2, 1, 100)
check("Mean pairwise agreement (%)", pairwise, 96.1, 1, 100)
check("Fleiss' kappa", kappa, 0.541, 3)
check("Prevalence-adjusted bias-adjusted kappa", pabak, 0.92, 2)

**What.** Majority-vote accuracy per model over every rated field, then per field (Table 1), with the
two models compared by an exact McNemar test on the paired majority votes.

**Why McNemar.** Both models were judged on the same submissions, so their accuracies are paired. The
McNemar test uses only the discordant pairs, the fields one model got right and the other wrong, which is
the information that distinguishes the two models. The exact binomial form is used because some fields
have few discordant pairs. The 19 tests are reported without multiplicity adjustment, as the paper
states.

In [ ]:
MODELS = {"stage1": "DeepSeek V4 Flash", "stage3": "GPT-5.6 Luna"}
for model, name in MODELS.items():
    units = [u for u in ratings if unit_model[u] == model]
    right = sum(majority[u] for u in units)
    print(f"{name}: {right:,} of {len(units):,} fields judged correct")
    check(
        f"{name}: fields judged correct (%)",
        right / len(units),
        {"stage1": 94.9, "stage3": 97.1}[model],
        1,
        100,
    )

TOP_LEVEL = [f for f in EXTRACTED_FIELDS]
rows = []
for field in TOP_LEVEL:
    by_record = {
        m: {unit_record[u]: majority[u] for u in ratings if unit_model[u] == m and u[1] == field}
        for m in MODELS
    }
    shared = [r for r in by_record["stage1"] if r in by_record["stage3"]]
    only_deepseek = sum(by_record["stage1"][r] and not by_record["stage3"][r] for r in shared)
    only_luna = sum(not by_record["stage1"][r] and by_record["stage3"][r] for r in shared)
    p = stats.binomtest(min(only_deepseek, only_luna), only_deepseek + only_luna, 0.5).pvalue
    rows.append(
        {
            "field": field,
            "n": len(shared),
            "DeepSeek %": 100 * sum(by_record["stage1"][r] for r in shared) / len(shared),
            "Luna %": 100 * sum(by_record["stage3"][r] for r in shared) / len(shared),
            "only DeepSeek right": only_deepseek,
            "only Luna right": only_luna,
            "McNemar P": p,
        }
    )
table2 = pd.DataFrame(rows)
display(table2.round({"DeepSeek %": 1, "Luna %": 1}))

self_use_row = table2.set_index("field").loc["a_self_use"]
check("Self-use accuracy, DeepSeek (%)", self_use_row["DeepSeek %"], 93.3, 1)
check("Self-use accuracy, GPT-5.6 Luna (%)", self_use_row["Luna %"], 99.2, 1)
check("Self-use discordant pairs favouring Luna", self_use_row["only Luna right"], 75)
products_row = table2.set_index("field").loc["b_products_mentioned"]
check("Product-list accuracy, GPT-5.6 Luna (%)", products_row["Luna %"], 99.3, 1)

**What.** Two details the paper reports from the same ratings: which way DeepSeek's self-use errors
went, and accuracy at the level of individual extracted side effects.

**Why.** A model that misses self-reports shrinks the analysis population, while one that invents them
adds noise; the direction of error matters for interpreting the counts.

In [ ]:
# Load the model outputs that were rated; only structured fields are read from them
model_values = {
    int(r["model_output_id"]): json.loads(r["extraction_json"])
    for r in csv.DictReader((PRIVATE / "model_outputs_export.csv").open(encoding="utf-8"))
}

wrong_self_use = [
    u for u in ratings if unit_model[u] == "stage1" and u[1] == "a_self_use" and not majority[u]
]
said_false = sum(model_values[u[0]].get("a_self_use") is False for u in wrong_self_use)
check("DeepSeek self-use decisions judged wrong", len(wrong_self_use), 78)
check("... of which the model said 'not self-use'", said_false, 54)

for part, printed in (
    ("raw_span", {"stage1": 87.4, "stage3": 90.0}),
    ("meddra_pt", {"stage1": 85.4, "stage3": 89.0}),
):
    for model in MODELS:
        units = [
            u
            for u in ratings
            if unit_model[u] == model and canonical(u[1]) == f"d_side_effect_pts[*].{part}"
        ]
        check(
            f"{MODELS[model]}: {part} judged correct (%)",
            sum(majority[u] for u in units) / len(units),
            printed[model],
            1,
            100,
        )

**What.** Agreement per field (Supplementary Table 4) and accuracy restricted to outputs where the model
extracted an informative value (Supplementary Table 5).

**Why.** The pooled accuracy scores every field on every submission, so sparse fields are judged mostly
on correct nulls. Restricting to informative values shows how reliable a field is when it actually
carries information. These conditional rates are precisions: they say nothing about values the model
missed.

In [ ]:
rows = []
for field in [*TOP_LEVEL, "d_side_effect_pts[*].raw_span", "d_side_effect_pts[*].meddra_pt"]:
    units = [u for u in ratings if canonical(u[1]) == field]
    c = [correct_counts[u] for u in units]
    row = {
        "field": field,
        "units": len(units),
        "pairwise %": 100 * pairwise_binary(c, RATERS),
        "unanimous %": 100 * sum(x in (0, RATERS) for x in c) / len(c),
        "Fleiss kappa": fleiss_binary(c, RATERS),
    }
    for model in MODELS:
        mu = [u for u in units if unit_model[u] == model]
        row[f"{MODELS[model]} %"] = 100 * sum(correct_counts[u] for u in mu) / (len(mu) * RATERS)
    rows.append(row)
display(pd.DataFrame(rows).round(3))


def informative(value):
    """A value is informative when it is not null and not an empty list or string."""
    return value is not None and not (isinstance(value, (list, str)) and len(value) == 0)


rows = []
for model in MODELS:
    for field in TOP_LEVEL[1:]:
        units = [
            u
            for u in ratings
            if unit_model[u] == model
            and u[1] == field
            and informative(model_values[u[0]].get(field))
        ]
        if units:
            right = sum(majority[u] for u in units)
            low, high = wilson(right, len(units))
            rows.append(
                {
                    "model": MODELS[model],
                    "field": field,
                    "n": len(units),
                    "correct": right,
                    "%": 100 * right / len(units),
                    "low": 100 * low,
                    "high": 100 * high,
                }
            )
conditional = pd.DataFrame(rows)
display(conditional.round(1))
luna_lists = conditional.query("model == 'GPT-5.6 Luna' and field == 'd_side_effect_pts'").iloc[0]
check("Luna non-empty side-effect lists judged correct", luna_lists["correct"], 108)
check("Luna non-empty side-effect lists", luna_lists["n"], 136)

**What.** Verification broken down by the indication group each model assigned (Supplementary Table 6).
For side-effect presence, a non-empty list with at least one span judged correct is a true positive, a
non-empty list with no correct span a false positive, an empty list judged incorrect (a missed side
effect) a false negative, and an empty list judged correct a true negative.

**Why.** The main comparison is between groups. If extraction were more sensitive in one group than the
other, a difference in reported prevalence could be produced by the extractor alone. Pooled accuracy
cannot show this; accuracy by group can.

In [ ]:
GROUPS3 = ("medical", "cosmetic", "unknown")

# One entry per rated output the model classified as self-use, in the order outputs first appear
span_verdicts = defaultdict(list)
for unit in ratings:
    if SPAN.fullmatch(unit[1]) and unit[1].endswith("raw_span"):
        span_verdicts[unit[0]].append(majority[unit])
by_group = defaultdict(list)
for output_id, model in output_model.items():
    value = model_values[output_id]
    if not value.get("a_self_use"):
        continue
    group = str(value.get("c_indication_category") or "unknown").casefold().strip()
    group = group if group in GROUPS3 else "unknown"
    by_group[(model, group)].append(
        {
            "indication_correct": majority.get((output_id, "c_indication_category")),
            "list_correct": majority.get((output_id, "d_side_effect_pts")),
            "listed": bool(value.get("d_side_effect_pts")),
            "any_span_correct": any(span_verdicts[output_id]),
        }
    )


def confusion(outputs):
    """True and false positives and negatives for side-effect presence."""
    return {
        "tp": sum(o["listed"] and o["any_span_correct"] for o in outputs),
        "fp": sum(o["listed"] and not o["any_span_correct"] for o in outputs),
        "fn": sum(not o["listed"] and o["list_correct"] is False for o in outputs),
        "tn": sum(not o["listed"] and o["list_correct"] is True for o in outputs),
    }


rows = []
for model in MODELS:
    for group in GROUPS3:
        outputs = by_group[(model, group)]
        c = confusion(outputs)
        rows.append(
            {
                "model": MODELS[model],
                "group": group,
                "n": len(outputs),
                "indication correct %": 100
                * sum(bool(o["indication_correct"]) for o in outputs)
                / len(outputs),
                **c,
                "sensitivity %": 100 * c["tp"] / (c["tp"] + c["fn"]),
                "PPV %": 100 * c["tp"] / (c["tp"] + c["fp"]),
            }
        )
by_arm = pd.DataFrame(rows)
display(by_arm.round(1))
luna = by_arm[by_arm["model"] == "GPT-5.6 Luna"].set_index("group")
check("Luna medical self-use outputs", luna.loc["medical", "n"], 206)
check("Luna cosmetic self-use outputs", luna.loc["cosmetic", "n"], 117)
check("Luna sensitivity, medical (%)", luna.loc["medical", "sensitivity %"], 98.5, 1)
check("Luna sensitivity, cosmetic (%)", luna.loc["cosmetic", "sensitivity %"], 92.7, 1)
check("Luna PPV, medical (%)", luna.loc["medical", "PPV %"], 90.4, 1)
check("Luna PPV, cosmetic (%)", luna.loc["cosmetic", "PPV %"], 95.0, 1)
check(
    "Luna 'unknown' indication judged correct (%)",
    luna.loc["unknown", "indication correct %"],
    50.0,
    1,
)

## 4. Extraction yield and clinical content

**What.** Descriptive results of the primary extraction (Results, second subsection; Figure 2;
Table 2): how many submissions describe personal use, how many report a side effect, the indication
split, how often each optional field was disclosed, the dose distribution, the most frequent candidate
terms, attitudes toward side effects, the products named, and how many authors wrote the submissions.

**Why.** These are the denominators every later analysis rests on, and the disclosure rates explain why
dose, provider, care-seeking and demographic fields are treated as descriptive only: they are too
sparse to adjust for.

In [ ]:
primary = views["stage4"]
self_use = [v for v in primary.values() if v["a_self_use"]]
reporting = [v for v in self_use if v["d_side_effect_pts"]]
check("Self-use submissions", len(self_use), 39_383)
check("Self-use share of extracted submissions (%)", len(self_use) / len(primary), 33.2, 1, 100)
check("Submissions reporting a candidate side effect", len(reporting), 14_120)
check("... as a share of all submissions (%)", len(reporting) / len(primary), 11.9, 1, 100)
check("... as a share of self-use submissions (%)", len(reporting) / len(self_use), 35.9, 1, 100)
check(
    "Submissions reporting more than one side effect",
    sum(len(v["d_side_effect_pts"]) > 1 for v in self_use),
    7_886,
)

indication = Counter(v["c_indication_category"] for v in self_use)
for group, n, share in (
    ("medical", 20_811, 52.8),
    ("cosmetic", 12_716, 32.3),
    ("unknown", 5_856, 14.9),
):
    check(f"{group} self-use submissions", indication[group], n)
    check(f"{group} share of self-use (%)", indication[group] / len(self_use), share, 1, 100)

attitude = Counter(v["f_sentiment_about_side_effect"] for v in reporting)
for label, share in (("distressed", 84.2), ("accepting", 15.0), ("dismissive", 0.7)):
    check(
        f"Side-effect reports expressing '{label}' (%)",
        attitude[label] / len(reporting),
        share,
        1,
        100,
    )

**What.** How often each optional field carried information among self-use submissions, the dose
distribution, and the candidate terms named most often.

**Why.** A field counts as disclosed when the model returned a value other than null; for provider,
onset and duration the catch-all value *unspecified*, and for discontinuation *unclear*, also count as
undisclosed. The median and interquartile range describe dose because its distribution is skewed.

In [ ]:
UNINFORMATIVE = {
    "c_indication_subtype": {None},
    "g_treatment_area_freetext": {None},
    "h_dose_units": {None},
    "i_provider_type": {None, "unspecified"},
    "j_first_time_or_repeat": {None},
    "k_time_to_onset": {None, "unspecified"},
    "m_sought_medical_attention": {None},
}
printed = {
    "c_indication_subtype": 84.8,
    "g_treatment_area_freetext": 54.4,
    "j_first_time_or_repeat": 39.5,
    "i_provider_type": 13.8,
    "k_time_to_onset": 12.9,
    "m_sought_medical_attention": 9.4,
    "h_dose_units": 8.5,
}
for field, share in printed.items():
    disclosed = sum(v[field] not in UNINFORMATIVE[field] for v in self_use)
    check(f"Disclosed: {field} (%)", disclosed / len(self_use), share, 1, 100)

doses = np.array(
    [v["h_dose_units"] for v in self_use if v["h_dose_units"] is not None], dtype=float
)
check("Self-use submissions stating a dose", len(doses), 3_364)
check("Median dose (units)", float(np.median(doses)), 50)
check("Dose lower quartile (units)", float(np.quantile(doses, 0.25)), 25)
check("Dose upper quartile (units)", float(np.quantile(doses, 0.75)), 100)

term_counts = Counter(t for v in self_use for t in v["d_side_effect_pts"])
for term, n in (
    ("dysphagia", 1_286),
    ("pain", 878),
    ("headache", 758),
    ("eructation", 731),
    ("injection site pain", 715),
):
    check(f"Self-use submissions naming '{term}'", term_counts[term], n)

**What.** Products named in self-use submissions (Table 2).

**Why.** The field is multi-label, so shares are computed against all self-use submissions and can sum to
more than 100%. Botox dominates because the trademark is used as the generic name of the drug, which
makes product counts a lower bound on the use of other brands.

In [ ]:
products = Counter(p for v in self_use for p in v["b_products_mentioned"])
table1 = pd.DataFrame(
    [
        {"product": p, "self-use submissions": n, "share %": round(100 * n / len(self_use), 1)}
        for p, n in products.most_common()
    ]
    + [
        {
            "product": "(none identifiable)",
            "self-use submissions": sum(not v["b_products_mentioned"] for v in self_use),
            "share %": None,
        }
    ]
)
display(suppress_small(table1, ["self-use submissions"]))
check("Self-use submissions naming Botox", products["botox"], 37_258)
check("Self-use submissions naming Dysport", products["dysport"], 1_578)

**What.** How many authors wrote the extracted submissions, and how concentrated their posting is.

**Why.** The analysis unit is the submission, not the person. If a few prolific authors wrote much of
the corpus, submissions would not be independent observations; these numbers motivate the
author-clustered analyses in Section 7. Only named authors are counted, and the share contributed by
repeat posters is computed among submissions that have a named author.

In [ ]:
posts_per_author = Counter(meta[rid][0] for rid in primary if meta[rid][0] is not None)
named_posts = sum(posts_per_author.values())
repeat_posts = sum(n for n in posts_per_author.values() if n > 1)
check("Named authors of extracted submissions", len(posts_per_author), 79_535)
check(
    "Authors with a single submission (%)",
    sum(n == 1 for n in posts_per_author.values()) / len(posts_per_author),
    78.6,
    1,
    100,
)
check(
    "Submissions by repeat posters, of those with a named author (%)",
    repeat_posts / named_posts,
    47.0,
    1,
    100,
)
print(
    f"Submissions by repeat posters, of all extracted submissions: {100 * repeat_posts / len(primary):.1f}%"
)
largest_share = max(posts_per_author.values()) / len(primary)
print(f"Largest single-account share of submissions: under {math.ceil(100 * largest_share)}%")
assert largest_share < 0.01, "the paper states that no account contributed more than 1%"

## 5. Agreement between extraction configurations (Supplementary Note 2)

**What.** Pairwise agreement between the four configurations on the records they share (Supplementary Table 2), the
agreement of the two full-corpus runs on the two gate decisions, the three-configuration summary, and
the operational summary (Supplementary Table 3).

**Why.** These comparisons measure internal consistency, not accuracy: how much a result depends on the
choice of model, reasoning setting or processing mode. For categorical fields, agreement is reported
as Cohen's kappa; for the multi-label term set, as the share of identical sets and the mean Jaccard
similarity. Downstream fields are compared only where both configurations classified the record as
self-use, so agreement is not inflated by fields both runs set to null. Comparisons involving the two
full-corpus runs are restricted to the nested samples, as in the analysis scripts; the whole-corpus
gate agreement is computed separately.

In [ ]:
retained = set(views["stage2"]) | set(views["stage3"])  # records of the two nested samples


def jaccard(a, b):
    """Jaccard similarity of two sets; two empty sets count as identical."""
    return 1.0 if not a and not b else len(a & b) / len(a | b)


def pair_agreement(left, right):
    """Supplementary Table 2 metrics for two configurations on their shared records."""
    lv, rv = views[left], views[right]
    ids = sorted(
        set(lv) & set(rv) & (retained if {left, right} & {"stage1", "stage4"} else set(lv))
    )
    both_self_use = [r for r in ids if lv[r]["a_self_use"] and rv[r]["a_self_use"]]
    return {
        "comparison": f"{left} vs {right}",
        "paired n": len(ids),
        "self-use kappa": cohen_kappa(
            [lv[r]["a_self_use"] for r in ids], [rv[r]["a_self_use"] for r in ids]
        ),
        "any-effect kappa": cohen_kappa(
            [lv[r]["side_effect_presence"] for r in ids],
            [rv[r]["side_effect_presence"] for r in ids],
        ),
        "term-set exact": np.mean(
            [lv[r]["d_side_effect_pts"] == rv[r]["d_side_effect_pts"] for r in both_self_use]
        ),
        "mean Jaccard": np.mean(
            [jaccard(lv[r]["d_side_effect_pts"], rv[r]["d_side_effect_pts"]) for r in both_self_use]
        ),
        "full record": np.mean([all(lv[r][f] == rv[r][f] for f in EXTRACTED_FIELDS) for r in ids]),
    }


table_b1 = pd.DataFrame(
    [
        pair_agreement(a, b)
        for a, b in (
            ("stage1", "stage2"),
            ("stage1", "stage3"),
            ("stage2", "stage3"),
            ("stage1", "stage4"),
            ("stage2", "stage4"),
            ("stage3", "stage4"),
        )
    ]
).set_index("comparison")
display(table_b1.round(3))

printed_b1 = {
    "stage1 vs stage2": (11_824, 0.871, 0.851, 0.701, 0.752, 0.521),
    "stage1 vs stage3": (1_168, 0.840, 0.806, 0.639, 0.687, 0.413),
    "stage2 vs stage3": (1_165, 0.848, 0.773, 0.640, 0.683, 0.412),
    "stage1 vs stage4": (11_825, 0.843, 0.808, 0.635, 0.684, 0.383),
    "stage2 vs stage4": (11_822, 0.854, 0.793, 0.641, 0.690, 0.414),
    "stage3 vs stage4": (1_168, 0.967, 0.955, 0.741, 0.788, 0.515),
}
for comparison, values in printed_b1.items():
    for column, value in zip(table_b1.columns, values, strict=True):
        check(
            f"Supplementary Table 2 {comparison}: {column}",
            table_b1.loc[comparison, column],
            value,
            0 if column == "paired n" else 3,
        )

**What.** Agreement of the two full-corpus runs on self-use and on the presence of any side effect, over
every record both completed; the three-configuration range of the key proportions; and the difference
in the number of self-use submissions the two models found.

**Why.** The two gate decisions define the analysis population, so their agreement across the whole
corpus, not only the samples, is the relevant check.

In [ ]:
shared = [r for r in views["stage1"] if r in views["stage4"]]
for field, agreement, kappa_printed in (
    ("a_self_use", 93.0, 0.839),
    ("side_effect_presence", 96.4, 0.817),
):
    a = [views["stage1"][r][field] for r in shared]
    b = [views["stage4"][r][field] for r in shared]
    check(
        f"Full-corpus agreement, {field} (%)",
        np.mean([x == y for x, y in zip(a, b, strict=True)]),
        agreement,
        1,
        100,
    )
    check(f"Full-corpus kappa, {field}", cohen_kappa(a, b), kappa_printed, 3)

common = set(views["stage1"]) & set(views["stage2"]) & set(views["stage3"])
check("Records completed by all three interactive configurations", len(common), 1_165)
self_use_shares = [
    np.mean([views[s][r]["a_self_use"] for r in common]) for s in ("stage1", "stage2", "stage3")
]
effect_shares = [
    np.mean([views[s][r]["side_effect_presence"] for r in common])
    for s in ("stage1", "stage2", "stage3")
]
check("Lowest self-use share across configurations (%)", min(self_use_shares), 30.5, 1, 100)
check("Highest self-use share across configurations (%)", max(self_use_shares), 33.0, 1, 100)
check("Lowest side-effect share across configurations (%)", min(effect_shares), 9.7, 1, 100)
check("Highest side-effect share across configurations (%)", max(effect_shares), 11.4, 1, 100)

extra = sum(v["a_self_use"] for v in views["stage4"].values()) - sum(
    v["a_self_use"] for v in views["stage1"].values()
)
check("Additional self-use submissions found by GPT-5.6 Luna", extra, 2_137)

**What.** The operational summary of the four runs (Supplementary Table 3): successes, failures, the share of
extracted spans that occur verbatim in the source, cost, and cost per 1,000 successes.

**Why.** Cost and failure behaviour were part of the model choice. Costs and token counts for the three
interactive runs come from the run summary written by the extraction runner; for the batch run they are
summed from the per-call usage stored in the checkpoint. The verbatim-span rate checks that the model
quoted rather than paraphrased when it extracted a side-effect span.

In [ ]:
run_summary = json.loads((PRIVATE / "run_summary.json").read_text())["stages"]
rows = []
for stage in ("stage1", "stage2", "stage3", "stage4"):
    if stage in run_summary:
        s = run_summary[stage]
        successes, cost = s["completed_records"], s["reported_cost_usd"]
        verbatim, cache = s["raw_span_verbatim_rate"], s["cache_hit_rate"]
    else:  # the batch run: sum the usage stored with each call
        successes, prompt, cached, cost, verbatim_n, verbatim_d = con.execute(
            """
            SELECT COUNT(*), SUM(json_extract(metadata_json, '$.usage.prompt_tokens')),
                   SUM(json_extract(metadata_json, '$.usage.prompt_tokens_details.cached_tokens')),
                   SUM(json_extract(metadata_json, '$.usage.cost')),
                   SUM(json_extract(metadata_json, '$.raw_span_metrics.verbatim_raw_spans')),
                   SUM(json_extract(metadata_json, '$.raw_span_metrics.total_raw_spans'))
            FROM stage_results WHERE stage = ?""",
            (stage,),
        ).fetchone()
        verbatim, cache = verbatim_n / verbatim_d, cached / prompt
    failures = con.execute(
        "SELECT COUNT(*) FROM stage_failures WHERE stage = ?", (stage,)
    ).fetchone()[0]
    rows.append(
        {
            "configuration": stage,
            "successes": successes,
            "failures": failures,
            "verbatim spans %": 100 * verbatim,
            "cache hit %": 100 * cache,
            "cost US$": cost,
            "per 1,000 US$": 1000 * cost / successes,
        }
    )
table_b2 = pd.DataFrame(rows).set_index("configuration")
display(table_b2.round(2))
for stage, verbatim, cost in (
    ("stage1", 94.2, 26.34),
    ("stage2", 92.7, 3.59),
    ("stage3", 99.0, 4.77),
    ("stage4", 98.6, 26.60),
):
    check(f"Verbatim spans, {stage} (%)", table_b2.loc[stage, "verbatim spans %"], verbatim, 1)
    check(f"Cost, {stage} (US$)", table_b2.loc[stage, "cost US$"], cost, 2)
check("Batch-run prompt-cache coverage (%)", table_b2.loc["stage4", "cache hit %"], 86.8, 1)

## 6. Medical and cosmetic reporting profiles

**What.** The primary comparison. The population is self-use submissions from the primary extraction
whose indication is medical or cosmetic; the 5,856 with an unknown indication are excluded. Two
quantities are compared:

1. Reporting prevalence, the share of submissions in each group that report at least one candidate
   side effect.
2. Reporting composition, the distribution of candidate terms among submissions that report at least
   one.

**Why two quantities.** They answer different questions and use different denominators. Prevalence
asks whether a side effect is mentioned at all; composition asks what is mentioned, among those who
mention something. Reddit gives no count of people treated, so neither is an incidence.

Records are kept in checkpoint order, because the permutation test below draws its permutations in that
order and must match the archived run exactly.

In [ ]:
@dataclass(frozen=True)
class Record:
    arm: str  # "medical" or "cosmetic"
    terms: frozenset  # the candidate terms of the submission (empty if none reported)


def comparison_records(stage_views):
    """Self-use records with a medical or cosmetic indication, in the order given."""
    return [
        Record(v["c_indication_category"], v["d_side_effect_pts"])
        for v in stage_views.values()
        if v["a_self_use"] and v["c_indication_category"] in ("medical", "cosmetic")
    ]


records = comparison_records(views["stage4"])
unknown = [
    v
    for v in views["stage4"].values()
    if v["a_self_use"] and v["c_indication_category"] == "unknown"
]
unknown_reporting = sum(v["side_effect_presence"] for v in unknown)
low, high = wilson(unknown_reporting, len(unknown))
check("Unknown-indication submissions reporting a side effect", unknown_reporting, 2_735)
check("... share (%)", unknown_reporting / len(unknown), 46.7, 1, 100)
check("... lower 95% bound (%)", low, 45.4, 1, 100)
check("... upper 95% bound (%)", high, 48.0, 1, 100)

**What.** Prevalence by group, the medical-minus-cosmetic difference with a Newcombe-Wilson interval,
the risk ratio with a Katz interval, and a Pearson chi-square test without continuity correction.

**Why.** The difference in percentage points is the directly interpretable effect; the risk ratio gives
the relative size. The continuity correction is omitted because it is conservative at this sample
size and the paper reports the uncorrected statistic. These intervals treat submissions as independent;
Section 7 relaxes that.

In [ ]:
def prevalence_test(recs):
    """Chi-square test, difference and risk ratio for side-effect reporting, medical versus cosmetic."""
    n = Counter(r.arm for r in recs)
    k = Counter(r.arm for r in recs if r.terms)
    table = np.array(
        [
            [k["medical"], n["medical"] - k["medical"]],
            [k["cosmetic"], n["cosmetic"] - k["cosmetic"]],
        ]
    )
    chi2, p, _dof, _ = stats.chi2_contingency(table, correction=False)
    difference = k["medical"] / n["medical"] - k["cosmetic"] / n["cosmetic"]
    low, high = newcombe(k["medical"], n["medical"], k["cosmetic"], n["cosmetic"])
    ratio, ratio_low, ratio_high = katz_risk_ratio(
        k["medical"], n["medical"], k["cosmetic"], n["cosmetic"]
    )
    return {
        "n_medical": n["medical"],
        "n_cosmetic": n["cosmetic"],
        "k_medical": k["medical"],
        "k_cosmetic": k["cosmetic"],
        "difference": difference,
        "low": low,
        "high": high,
        "chi2": chi2,
        "p": p,
        "risk_ratio": ratio,
        "rr_low": ratio_low,
        "rr_high": ratio_high,
    }


primary_test = prevalence_test(records)
for arm, k, _n, share in (("medical", 6_847, 20_811, 32.9), ("cosmetic", 4_538, 12_716, 35.7)):
    check(f"{arm} submissions reporting a side effect", primary_test[f"k_{arm}"], k)
    check(
        f"{arm} prevalence (%)", primary_test[f"k_{arm}"] / primary_test[f"n_{arm}"], share, 1, 100
    )
check("Prevalence difference (percentage points)", primary_test["difference"], -2.79, 2, 100)
check("... lower 95% bound", primary_test["low"], -3.84, 2, 100)
check("... upper 95% bound", primary_test["high"], -1.74, 2, 100)
check("Risk ratio", primary_test["risk_ratio"], 0.92, 2)
check("Pearson chi-square", primary_test["chi2"], 27.3, 1)
print(f"P = {primary_test['p']:.2g}")
check("Chi-square P (x 10^-7)", primary_test["p"] * 1e7, 1.7, 1)

**What.** The omnibus test of composition: the base-2 Jensen-Shannon divergence between the medical and
cosmetic term profiles, tested by permuting the indication labels 10,000 times among the submissions
that report a side effect.

**Why this test.** One submission can report several terms, so term counts are not independent and a
chi-square test on the term-by-group table would be invalid. Permuting labels at the submission level
keeps each submission's set of terms together, so the null distribution preserves the correlation
among terms within a submission. Group sizes are held fixed. The primary test uses the 221 terms
present in at least 20 submissions, because very rare terms add noise that inflates the divergence
(the all-term version is reported as a sensitivity analysis). The P value is the upper-tail share of
permutations at least as large as the observed divergence, with the plus-one correction, so the
smallest attainable value is 1/10,001.

In [ ]:
def profile_permutation_test(
    recs, min_count=MIN_COUNT, permutations=PERMUTATIONS, seed=PERMUTATION_SEED, batch=100
):
    """Jensen-Shannon divergence between the two term profiles, with a label-permutation P value."""
    reporting_recs = [r for r in recs if r.terms]
    pooled = Counter(t for r in reporting_recs for t in r.terms)
    terms = sorted(pooled)
    index = {t: i for i, t in enumerate(terms)}
    rows_, cols_ = zip(
        *[(i, index[t]) for i, r in enumerate(reporting_recs) for t in r.terms], strict=True
    )
    # one row per reporting submission, one column per term, 1 where the submission names the term
    X = sparse.csr_matrix(
        (np.ones(len(rows_)), (rows_, cols_)), shape=(len(reporting_recs), len(terms))
    )
    is_medical = np.array([r.arm == "medical" for r in reporting_recs], dtype=np.uint8)
    medical_counts = np.asarray(is_medical.astype(float) @ X).ravel()
    total_counts = np.asarray(X.sum(axis=0)).ravel()
    masks = {
        "all": np.ones(len(terms), dtype=bool),
        "common": np.array([pooled[t] >= min_count for t in terms]),
    }
    observed = {
        k: float(jsd(medical_counts[m], (total_counts - medical_counts)[m])[0])
        for k, m in masks.items()
    }
    exceed = dict.fromkeys(masks, 0)
    null_sum = dict.fromkeys(masks, 0.0)
    null_sq = dict.fromkeys(masks, 0.0)
    rng = np.random.default_rng(seed)
    for start in range(0, permutations, batch):
        size = min(batch, permutations - start)
        # shuffle the labels independently in each of `size` copies
        shuffled = rng.permuted(np.broadcast_to(is_medical, (size, len(is_medical))), axis=1)
        perm_medical = np.asarray(shuffled.astype(float) @ X)
        perm_cosmetic = total_counts[None, :] - perm_medical
        for k, m in masks.items():
            null = jsd(perm_medical[:, m], perm_cosmetic[:, m])
            exceed[k] += int(np.count_nonzero(null >= observed[k] - 1e-15))
            null_sum[k] += float(null.sum())
            null_sq[k] += float(np.square(null).sum())
    out = {}
    for k, m in masks.items():
        mean = null_sum[k] / permutations
        sd = math.sqrt(max(0.0, (null_sq[k] - permutations * mean * mean) / (permutations - 1)))
        out[k] = {
            "jsd": observed[k],
            "terms": int(m.sum()),
            "p": (exceed[k] + 1) / (permutations + 1),
            "null_mean": mean,
            "null_sd": sd,
        }
    return out


profile = profile_permutation_test(records)
print(profile)
check("Terms in the primary profile test", profile["common"]["terms"], 221)
check("Jensen-Shannon divergence", profile["common"]["jsd"], 0.372, 3)
check("Permutation-null mean", profile["common"]["null_mean"], 0.009, 3)
check("Permutation P (x 10^-4)", profile["common"]["p"] * 1e4, 1.0, 1)
check("Terms in the all-term test", profile["all"]["terms"], 3_075)
check("All-term divergence", profile["all"]["jsd"], 0.454, 3)
check("All-term null mean", profile["all"]["null_mean"], 0.115, 3)

**What.** The number of distinct candidate terms per side-effect report, compared with a two-sided
Mann-Whitney test, and the rank-biserial correlation as its effect size.

**Why.** Counts per report are small integers with many ties, so a rank test is used rather than a
t-test. The rank-biserial correlation (twice U divided by the product of the group sizes, minus one)
expresses the effect on a -1 to 1 scale; means are also given because the medians are identical.

In [ ]:
medical_n_terms = np.array(
    [len(r.terms) for r in records if r.arm == "medical" and r.terms], dtype=float
)
cosmetic_n_terms = np.array(
    [len(r.terms) for r in records if r.arm == "cosmetic" and r.terms], dtype=float
)
mw = stats.mannwhitneyu(
    medical_n_terms, cosmetic_n_terms, alternative="two-sided", method="asymptotic"
)
rank_biserial = 2 * mw.statistic / (len(medical_n_terms) * len(cosmetic_n_terms)) - 1
check("Mann-Whitney U", mw.statistic, 16_861_852.5, 1)
check("Rank-biserial correlation", rank_biserial, 0.085, 3)
check("Mean terms per medical report", medical_n_terms.mean(), 2.36, 2)
check("Mean terms per cosmetic report", cosmetic_n_terms.mean(), 2.09, 2)
check(
    "Median terms per report (both groups)", np.median(np.r_[medical_n_terms, cosmetic_n_terms]), 2
)
check("Medical reports naming a single term (%)", np.mean(medical_n_terms == 1), 42.2, 1, 100)
check("Cosmetic reports naming a single term (%)", np.mean(cosmetic_n_terms == 1), 48.7, 1, 100)

**What.** Each of the 221 terms tested on its own: the difference in reports per 100 side-effect
reports between groups, a two-sided Fisher exact test, and Benjamini-Hochberg control of the false
discovery rate at 5%. The eight terms with the largest absolute differences form Table 3.

**Why.** The omnibus test says the profiles differ; the per-term tests say where. Fisher's exact test is
used because many terms are rare in one group. With 221 tests, the Benjamini-Hochberg procedure keeps
the expected share of false discoveries at 5% without the loss of power of a family-wise correction.
The denominators are the side-effect-reporting submissions in each group, the same as in Figure 3.

In [ ]:
report_n = Counter(r.arm for r in records if r.terms)
term_n = {
    arm: Counter(t for r in records if r.arm == arm for t in r.terms)
    for arm in ("medical", "cosmetic")
}
pooled = term_n["medical"] + term_n["cosmetic"]
tested = sorted((t for t, n in pooled.items() if n >= MIN_COUNT), key=lambda t: (-pooled[t], t))
rows = []
for term in tested:
    a, c = term_n["medical"][term], term_n["cosmetic"][term]
    p = stats.fisher_exact([[a, report_n["medical"] - a], [c, report_n["cosmetic"] - c]]).pvalue
    rows.append(
        {
            "term": term,
            "medical per 100": 100 * a / report_n["medical"],
            "cosmetic per 100": 100 * c / report_n["cosmetic"],
            "medical n": a,
            "cosmetic n": c,
            "p": p,
        }
    )
per_term = pd.DataFrame(rows)
per_term["difference"] = per_term["medical per 100"] - per_term["cosmetic per 100"]
per_term["reject"], per_term["q"] = multipletests(per_term["p"], alpha=0.05, method="fdr_bh")[:2]
check("Terms tested", len(per_term), 221)
check("Terms significant at a 5% false discovery rate", int(per_term["reject"].sum()), 163)

table3 = per_term.reindex(per_term["difference"].abs().sort_values(ascending=False).index).head(8)
display(table3[["term", "medical per 100", "cosmetic per 100", "difference", "q"]].round(2))
for term, difference in (
    ("dysphagia", 11.20),
    ("pain", 8.61),
    ("migraine", 5.33),
    ("facial asymmetry", -5.09),
    ("eyelid ptosis", -4.76),
    ("eructation", 4.37),
    ("nausea", 4.29),
    ("eyebrow ptosis", -4.23),
):
    check(
        f"Difference per 100 reports, {term}",
        per_term.set_index("term").loc[term, "difference"],
        difference,
        2,
    )

## 7. Sensitivity and robustness analyses

This section repeats the comparison under changed assumptions (Figure 4 and Supplementary Note 4). Each analysis
targets one specific threat to the two findings:

| Threat | Analysis |
|---|---|
| Repeat posters make submissions non-independent | one submission per author; author-clustered bootstrap; 200 random author draws |
| One community drives the result | leaving out each of the 20 largest communities, and groups of communities |
| The corpus changed over time | calendar periods, including a split in mid-2023 |
| The terms are unvalidated model proposals | removing indication-shaped terms; merging synonyms; a symmetric exclusion |
| The result depends on the model | the same comparison on the comparator model's extraction |
| Extraction errs differently by group | correcting prevalence for the measured sensitivity and predictive value |
| The excluded unknown-indication group | assigning all of it to one group or the other |
| The difference is only injection site | comparing medical and cosmetic uses injected in the same region |

A shared helper runs the prevalence test and the profile permutation test for any set of records.

In [ ]:
def summarise(name, recs, permutations=PERMUTATIONS, min_count=MIN_COUNT):
    """Prevalence difference and profile divergence for one variant of the comparison."""
    prevalence = prevalence_test(recs)
    divergence = profile_permutation_test(recs, min_count=min_count, permutations=permutations)[
        "common"
    ]
    return {
        "variant": name,
        "n medical": prevalence["n_medical"],
        "n cosmetic": prevalence["n_cosmetic"],
        "difference pp": 100 * prevalence["difference"],
        "low": 100 * prevalence["low"],
        "high": 100 * prevalence["high"],
        "P": prevalence["p"],
        "divergence": divergence["jsd"],
        "terms": divergence["terms"],
        "divergence P": divergence["p"],
    }

### 7.1 The three original sensitivity analyses

**What.** (i) One submission per named author, the lexicographically smallest record identifier, with
submissions whose author cannot be linked kept as single observations; (ii) excluding r/noburp;
(iii) removing seven terms that restate an indication or a lack of effect and merging the ptosis
variants, and regurgitation with reflux.

**Why.** Repeat posters supply nearly half of the submissions with a named author; r/noburp is a large,
unusual block of medical-use submissions organised around one outcome; and some candidate terms describe
the indication or treatment failure rather than an adverse effect. The records carry their author and
community from Section 1.

In [ ]:
# Checkpoint-ordered records with their record id, author and community
keyed = [
    (rid, Record(v["c_indication_category"], v["d_side_effect_pts"]))
    for rid, v in views["stage4"].items()
    if v["a_self_use"] and v["c_indication_category"] in ("medical", "cosmetic")
]

# (i) one submission per named author: the smallest record id; unlinkable authors kept as they are
by_author, unattributable = {}, []
for rid, rec in keyed:
    author = meta[rid][0]
    if author is None:
        unattributable.append(rec)
    elif author not in by_author or rid < by_author[author][0]:
        by_author[author] = (rid, rec)
author_dedup = [rec for _rid, rec in by_author.values()] + unattributable

# (ii) without r/noburp
no_noburp = [rec for rid, rec in keyed if meta[rid][1] != "noburp"]

# (iii) seven indication- or failure-shaped terms removed, two groups of near-synonyms merged
EXCLUDED = {
    "migraine",
    "migraine aggravated",
    "condition aggravated",
    "pain aggravated",
    "treatment failure",
    "drug ineffective",
    "therapeutic response decreased",
}
MERGED = {
    "ptosis": "ptosis (any)",
    "eyelid ptosis": "ptosis (any)",
    "eyebrow ptosis": "ptosis (any)",
    "brow ptosis": "ptosis (any)",
    "regurgitation": "reflux or regurgitation",
    "gastroesophageal reflux disease": "reflux or regurgitation",
    "gastrooesophageal reflux disease": "reflux or regurgitation",
}
term_validity = [
    Record(r.arm, frozenset(MERGED.get(t, t) for t in r.terms if t not in EXCLUDED))
    for _rid, r in keyed
]

original = pd.DataFrame(
    [
        summarise("one per author (smallest id)", author_dedup),
        summarise("without r/noburp", no_noburp),
        summarise("terms excluded and merged", term_validity),
    ]
).set_index("variant")
display(original.round(3))
for variant, difference, divergence, terms in (
    ("one per author (smallest id)", -1.49, 0.359, 174),
    ("without r/noburp", -8.14, 0.273, 191),
    ("terms excluded and merged", -3.85, 0.365, 212),
):
    check(f"{variant}: difference (pp)", original.loc[variant, "difference pp"], difference, 2)
    check(f"{variant}: divergence", original.loc[variant, "divergence"], divergence, 3)
    check(f"{variant}: terms", original.loc[variant, "terms"], terms)

### 7.2 Author-clustered inference

**What.** A bootstrap that resamples authors rather than submissions: each of 2,000 replicates draws
authors with replacement and takes all their submissions, and records whose author cannot be linked
form clusters of one. It gives intervals for the prevalence difference, the divergence and every
per-term difference; the per-term Benjamini-Hochberg count is then repeated with the clustered standard
errors. A second analysis draws one submission per author at random, 200 times.

**Why.** Submissions by the same author are likely to resemble one another, so treating them as
independent makes intervals too narrow. The cluster bootstrap estimates the variance with that
dependence left in. The random draws replace the single, arbitrary lexicographic choice of Section 7.1
with its whole distribution. From here on, records follow the order of the corpus file, as in the
robustness script, because the resampling draws depend on record order.

In [ ]:
@dataclass(frozen=True)
class Rec:
    arm: str
    terms: frozenset
    subtype: str
    author: str | None
    community: str
    created: float | None


primary_views = views["stage4"]
recs = [
    Rec(
        v["c_indication_category"],
        v["d_side_effect_pts"],
        v["c_indication_subtype"] or "",
        *meta[rid],
    )
    for rid in meta
    if (v := primary_views.get(rid)) is not None
    and v["a_self_use"]
    and v["c_indication_category"] in ("medical", "cosmetic")
]
rng = np.random.default_rng(RESAMPLING_SEED)  # shared, in this order, by 7.2 and 7.3

common_terms = sorted(
    t for t, n in Counter(t for r in recs for t in r.terms).items() if n >= MIN_COUNT
)
term_index = {t: i for i, t in enumerate(common_terms)}

# Assign clusters: one per named author, one per record without a linkable author
cluster_of, cluster_idx = {}, np.empty(len(recs), dtype=np.int64)
for i, r in enumerate(recs):
    cluster_idx[i] = cluster_of.setdefault(
        r.author if r.author is not None else f"\0single{i}", len(cluster_of)
    )
C = len(cluster_of)
is_med = np.array([r.arm == "medical" for r in recs], dtype=float)
has_se = np.array([bool(r.terms) for r in recs], dtype=float)
rr, cc = zip(
    *[(i, term_index[t]) for i, r in enumerate(recs) for t in r.terms if t in term_index],
    strict=True,
)
X = sparse.csr_matrix((np.ones(len(rr)), (rr, cc)), shape=(len(recs), len(common_terms)))
A = sparse.csr_matrix(
    (np.ones(len(recs)), (cluster_idx, np.arange(len(recs)))), shape=(C, len(recs))
)
# cluster totals: submissions, reporting submissions and term counts, by group
cl = {
    "med_n": A @ is_med,
    "cos_n": A @ (1 - is_med),
    "med_se": A @ (is_med * has_se),
    "cos_se": A @ ((1 - is_med) * has_se),
    "med_t": (A @ sparse.diags(is_med) @ X).tocsc(),
    "cos_t": (A @ sparse.diags(1 - is_med) @ X).tocsc(),
}


def cluster_statistics(w):
    """Prevalence difference, divergence and per-term differences for cluster weights w (replicates x clusters)."""
    med_se, cos_se = w @ cl["med_se"], w @ cl["cos_se"]
    med_t, cos_t = np.asarray((cl["med_t"].T @ w.T).T), np.asarray((cl["cos_t"].T @ w.T).T)
    prevalence = med_se / (w @ cl["med_n"]) - cos_se / (w @ cl["cos_n"])
    per_term = 100 * (med_t / med_se[:, None] - cos_t / cos_se[:, None])
    return prevalence, jsd(med_t, cos_t), per_term


observed_prev, observed_jsd, observed_terms = cluster_statistics(np.ones((1, C)))
boot_prev, boot_jsd, boot_terms = [], [], []
for _start in range(0, 2_000, 50):
    draws = rng.integers(0, C, size=(50, C))  # authors drawn with replacement
    w = np.vstack([np.bincount(d, minlength=C) for d in draws]).astype(float)
    p_, j_, t_ = cluster_statistics(w)
    boot_prev.append(p_)
    boot_jsd.append(j_)
    boot_terms.append(t_)
boot_prev, boot_jsd, boot_terms = (
    np.concatenate(boot_prev),
    np.concatenate(boot_jsd),
    np.vstack(boot_terms),
)

p_med, p_cos = cl["med_se"].sum() / cl["med_n"].sum(), cl["cos_se"].sum() / cl["cos_n"].sum()
naive_se = math.sqrt(
    p_med * (1 - p_med) / cl["med_n"].sum() + p_cos * (1 - p_cos) / cl["cos_n"].sum()
)
design_effect = (boot_prev.std(ddof=1) / naive_se) ** 2
clustered_p = 2 * stats.norm.sf(abs(observed_prev[0]) / boot_prev.std(ddof=1))
check("Clustered 95% interval, lower (pp)", np.percentile(boot_prev, 2.5), -4.08, 2, 100)
check("Clustered 95% interval, upper (pp)", np.percentile(boot_prev, 97.5), -1.44, 2, 100)
check("Design effect", design_effect, 1.59, 2)
check("Clustered P (x 10^-5)", clustered_p * 1e5, 3.6, 1)
check("Clustered divergence interval, lower", np.percentile(boot_jsd, 2.5), 0.363, 3)
check("Clustered divergence interval, upper", np.percentile(boot_jsd, 97.5), 0.403, 3)

# Per-term tests with clustered standard errors
se_terms = boot_terms.std(axis=0, ddof=1)
z_terms = observed_terms[0] / se_terms
reject = multipletests(2 * stats.norm.sf(np.abs(z_terms)), alpha=0.05, method="fdr_bh")[0]
check("Terms significant with clustered standard errors", int(reject.sum()), 159)

**What.** 200 random draws of one submission per named author (unlinkable records kept in every
draw), each giving a prevalence difference and a divergence over the same 221 terms.

**Why.** Any single draw is arbitrary; the distribution over draws shows how much the choice matters.

In [ ]:
groups = defaultdict(list)
singles = []
for i, r in enumerate(recs):
    (groups[r.author] if r.author is not None else singles).append(i)
author_groups = list(groups.values())
draw_diffs, draw_jsds = [], []
common_set = set(common_terms)
for _ in range(200):
    chosen = singles + [g[int(rng.integers(len(g)))] for g in author_groups]
    subset = [Record(recs[i].arm, recs[i].terms) for i in chosen]
    draw_diffs.append(prevalence_test(subset)["difference"])
    med = Counter(t for r in subset if r.arm == "medical" for t in r.terms & common_set)
    cos = Counter(t for r in subset if r.arm == "cosmetic" for t in r.terms & common_set)
    draw_jsds.append(float(jsd([med[t] for t in common_terms], [cos[t] for t in common_terms])[0]))
check("Random author draws: median difference (pp)", np.median(draw_diffs), -1.93, 2, 100)
check("... 2.5th percentile (pp)", np.percentile(draw_diffs, 2.5), -2.29, 2, 100)
check("... 97.5th percentile (pp)", np.percentile(draw_diffs, 97.5), -1.54, 2, 100)
check("... median divergence", np.median(draw_jsds), 0.367, 3)
print(f"negative in {100 * np.mean(np.array(draw_diffs) < 0):.0f}% of draws")

### 7.3 Leaving communities out

**What.** The comparison repeated without each of the 20 largest communities in turn (2,000
permutations each), and without four groups of communities (10,000 permutations each): the
harm-focused support community r/BotoxSupportCommunity, the two self-injection communities, all Reddit
user-profile pages, and r/noburp together with r/BotoxSupportCommunity.

**Why.** Reddit communities differ in who posts and why. If one community drives a finding, removing it
should change the finding. Removing r/noburp and r/BotoxSupportCommunity together takes out one
outcome-focused community from each side. User-profile pages are named after their owners, so they are
only ever handled as a group.

In [ ]:
def display_name(community):
    return "user-profile page" if community.startswith("u_") else f"r/{community}"


def as_records(subset):
    return [Record(r.arm, r.terms) for r in subset]


sizes = Counter(r.community for r in recs)
leave_out = []
for community, _n in sizes.most_common(20):
    removed = [r for r in recs if r.community == community]
    row = summarise(
        f"without {display_name(community)}",
        as_records(r for r in recs if r.community != community),
        permutations=STRATUM_PERMUTATIONS,
    )
    row["removed medical"] = sum(r.arm == "medical" for r in removed)
    row["removed cosmetic"] = sum(r.arm == "cosmetic" for r in removed)
    leave_out.append(row)
families = {
    "without r/botoxsupportcommunity (family)": lambda c: c == "botoxsupportcommunity",
    "without the DIY-injection communities": lambda c: (
        c in {"diyaesthetics", "diycosmeticprocedures"}
    ),
    "without user-profile pages": lambda c: c.startswith("u_"),
    "without r/noburp and r/botoxsupportcommunity": lambda c: (
        c in {"noburp", "botoxsupportcommunity"}
    ),
}
for name, test in families.items():
    leave_out.append(summarise(name, as_records(r for r in recs if not test(r.community))))
leave_out = pd.DataFrame(leave_out).set_index("variant")
display(leave_out.round(3))

check(
    "Without r/migraine: difference (pp)",
    leave_out.loc["without r/migraine", "difference pp"],
    -0.04,
    2,
)
check(
    "Without r/botoxsupportcommunity: difference (pp)",
    leave_out.loc["without r/botoxsupportcommunity", "difference pp"],
    -0.25,
    2,
)
check(
    "Without r/noburp and r/botoxsupportcommunity: difference (pp)",
    leave_out.loc["without r/noburp and r/botoxsupportcommunity", "difference pp"],
    -5.71,
    2,
)
singles_only = leave_out.iloc[:20]
check(
    "Lowest divergence leaving out one of the 20 largest",
    singles_only["divergence"].min(),
    0.273,
    3,
)
check(
    "Highest divergence leaving out one of the 20 largest",
    singles_only["divergence"].max(),
    0.408,
    3,
)
others = singles_only.drop(
    ["without r/migraine", "without r/noburp", "without r/botoxsupportcommunity"]
)
check("Other 17 communities: lowest difference (pp)", others["difference pp"].min(), -3.71, 2)
check("Other 17 communities: highest difference (pp)", others["difference pp"].max(), -1.13, 2)

### 7.4 Calendar periods

**What.** The comparison within 2015-2019, 2020-2022 and 2023-2025, and before and after July 2023.

**Why.** The corpus grows about thirty-fold over the decade, and around mid-2023 collection of the
archive passed from Pushshift to Arctic Shift. A finding that held only in one period would reflect the
data source rather than the use of the drug.

In [ ]:
CHANGE = datetime(2023, 7, 1, tzinfo=UTC).timestamp()


def year(r):
    return datetime.fromtimestamp(r.created, tz=UTC).year if r.created else 0


periods = {
    "2015-2019": lambda r: 0 < year(r) <= 2019,
    "2020-2022": lambda r: 2020 <= year(r) <= 2022,
    "2023-2025": lambda r: year(r) >= 2023,
    "before July 2023": lambda r: r.created is not None and r.created < CHANGE,
    "from July 2023": lambda r: r.created is not None and r.created >= CHANGE,
}
period_table = pd.DataFrame(
    [
        summarise(name, as_records(r for r in recs if test(r)), permutations=STRATUM_PERMUTATIONS)
        for name, test in periods.items()
    ]
).set_index("variant")
display(period_table.round(3))
check(
    "Lowest divergence across the three periods",
    period_table.iloc[:3]["divergence"].min(),
    0.347,
    3,
)
check(
    "Highest divergence across the three periods",
    period_table.iloc[:3]["divergence"].max(),
    0.383,
    3,
)

### 7.5 Comparing within the same injection region

**What.** Medical and cosmetic submissions whose indication subtype puts the injection in the same
region: chronic migraine against forehead and glabellar lines, temporomandibular disorders against
masseter and jawline treatment, blepharospasm against lateral canthal lines, and cervical dystonia
against neck and platysma treatment. The term threshold is lowered to 10 because these strata are
small.

**Why.** Medical and cosmetic injections go to different places, and the side effects of an injection
depend on where it is given. If the composition difference were only a difference of site, it would
vanish when site is held roughly constant.

In [ ]:
PAIRS = {
    "chronic migraine vs forehead or glabellar lines": (
        {"chronic_migraine"},
        {"forehead", "glabellar"},
    ),
    "blepharospasm vs lateral canthal lines": ({"blepharospasm"}, {"crows_feet"}),
    "temporomandibular disorder vs masseter or jawline": ({"tmj_medical"}, {"masseter_jawline"}),
    "cervical dystonia vs neck or platysma": ({"cervical_dystonia"}, {"neck_platysma"}),
}
site_rows = []
for name, (medical_subtypes, cosmetic_subtypes) in PAIRS.items():
    subset = as_records(
        r
        for r in recs
        if (r.arm == "medical" and r.subtype in medical_subtypes)
        or (r.arm == "cosmetic" and r.subtype in cosmetic_subtypes)
    )
    try:
        site_rows.append(
            summarise(name, subset, permutations=STRATUM_PERMUTATIONS, min_count=MIN_COUNT_SMALL)
        )
    except ValueError:  # no term reaches the threshold in one of the two groups
        site_rows.append(
            {
                "variant": name,
                "n medical": sum(r.arm == "medical" for r in subset),
                "n cosmetic": sum(r.arm == "cosmetic" for r in subset),
            }
        )
within_site = pd.DataFrame(site_rows).set_index("variant")
display(within_site.round(3))
check(
    "Upper face: divergence",
    within_site.loc["chronic migraine vs forehead or glabellar lines", "divergence"],
    0.297,
    3,
)
check(
    "Upper face: difference (pp)",
    within_site.loc["chronic migraine vs forehead or glabellar lines", "difference pp"],
    -22.05,
    2,
)
check(
    "Jaw: divergence",
    within_site.loc["temporomandibular disorder vs masseter or jawline", "divergence"],
    0.123,
    3,
)
check(
    "Jaw: difference (pp)",
    within_site.loc["temporomandibular disorder vs masseter or jawline", "difference pp"],
    -11.61,
    2,
)

### 7.6 Vocabulary

**What.** Two further term variants, applied identically to both groups: merging spelling variants and
synonyms of the same complaint (for example *muscle weakness* and *muscular weakness*), and, after
merging, also removing terms that name the treated condition, the intended effect of treatment (such as
*eructation* after cricopharyngeal injection) or a lack of effect.

**Why.** The candidate terms are model proposals, not coded MedDRA terms, so the same complaint can be
split across several labels, and some labels are not adverse effects at all. Applying the rules to both
groups at once avoids tilting the comparison. The word lists are imported from the robustness script so
that there is a single definition, and printed here in full.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "robustness_lists", ANALYSIS / "08_robustness_analyses.py"
)
robustness_lists = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = robustness_lists  # dataclasses need the module registered
spec.loader.exec_module(robustness_lists)
SYNONYMS, SYMMETRIC_EXCLUSIONS = robustness_lists.SYNONYMS, robustness_lists.SYMMETRIC_EXCLUSIONS
print("Synonym merges:", SYNONYMS)
print("Excluded from both groups:", sorted(SYMMETRIC_EXCLUSIONS))

merged = [Record(r.arm, frozenset(SYNONYMS.get(t, t) for t in r.terms)) for r in recs]
merged_excluded = [
    Record(r.arm, frozenset(t for t in r.terms if t not in SYMMETRIC_EXCLUSIONS)) for r in merged
]
vocabulary = pd.DataFrame(
    [
        summarise("synonyms merged", merged),
        summarise("synonyms merged, indication terms excluded", merged_excluded),
    ]
).set_index("variant")
display(vocabulary.round(3))
check("Synonyms merged: divergence", vocabulary.iloc[0]["divergence"], 0.367, 3)
check("Synonyms merged: terms", vocabulary.iloc[0]["terms"], 180)
check("Merged and excluded: divergence", vocabulary.iloc[1]["divergence"], 0.338, 3)
check("Merged and excluded: terms", vocabulary.iloc[1]["terms"], 162)
check("Merged and excluded: difference (pp)", vocabulary.iloc[1]["difference pp"], -3.41, 2)

### 7.7 The comparator model's extraction

**What.** The whole primary comparison on DeepSeek V4 Flash's full-corpus extraction instead of GPT-5.6
Luna's.

**Why.** If a finding appeared with one model only, it would reflect that model's habits. The comparator
classifies self-use and indication on its own, so its group sizes differ.

In [ ]:
deepseek_records = comparison_records(views["stage1"])
deepseek = summarise("comparator model", deepseek_records)
check("Comparator: medical self-use submissions", deepseek["n medical"], 21_010)
check("Comparator: cosmetic self-use submissions", deepseek["n cosmetic"], 12_955)
check("Comparator: difference (pp)", deepseek["difference pp"], -1.30, 2)
check("Comparator: lower bound (pp)", deepseek["low"], -2.33, 2)
check("Comparator: upper bound (pp)", deepseek["high"], -0.28, 2)
check("Comparator: divergence", deepseek["divergence"], 0.351, 3)
check("Comparator: terms", deepseek["terms"], 189)

### 7.8 Correcting for extraction error by group, and scenarios for the unknown group

**What.** (i) Each group's prevalence is corrected for the sensitivity and positive predictive value of
side-effect detection measured in that group in Section 3, here under the verification protocol's strict
rule (the paper's primary, post-level version is in section 7.9): corrected prevalence equals observed
prevalence multiplied by the predictive value and divided by the sensitivity. A bootstrap over the
verified outputs, 2,000 replicates, gives the interval. (ii) The 5,856 unknown-indication submissions
are assigned entirely to one group, then entirely to the other.

**Why.** (i) If the model missed more side effects in one group, the observed difference would be biased;
the correction removes the bias to the extent the verification sample can measure it. (ii) The excluded
group reports side effects more often than either analysed group, so where its members really belong can
move the difference; the two extreme assignments are scenarios that show how far it can move the difference; they are not
bounds, and section 7.9 adds scenarios that use the data.

In [ ]:
observed = {
    arm: primary_test[f"k_{arm}"] / primary_test[f"n_{arm}"] for arm in ("medical", "cosmetic")
}


def correction_factor(outputs):
    """Positive predictive value divided by sensitivity, from the verified outputs of one group."""
    c = confusion(outputs)
    return (c["tp"] / (c["tp"] + c["fp"])) / (c["tp"] / (c["tp"] + c["fn"]))


luna_outputs = {arm: by_group[("stage3", arm)] for arm in ("medical", "cosmetic")}
corrected = {arm: observed[arm] * correction_factor(luna_outputs[arm]) for arm in observed}
boot_rng = np.random.default_rng(RESAMPLING_SEED)
boot = []
for _ in range(2_000):
    resampled = {
        arm: [
            luna_outputs[arm][i]
            for i in boot_rng.integers(0, len(luna_outputs[arm]), len(luna_outputs[arm]))
        ]
        for arm in ("medical", "cosmetic")
    }
    try:
        boot.append(
            observed["medical"] * correction_factor(resampled["medical"])
            - observed["cosmetic"] * correction_factor(resampled["cosmetic"])
        )
    except ZeroDivisionError:
        continue
check("Corrected medical prevalence (%)", corrected["medical"], 30.2, 1, 100)
check("Corrected cosmetic prevalence (%)", corrected["cosmetic"], 36.6, 1, 100)
check("Corrected difference (pp)", corrected["medical"] - corrected["cosmetic"], -6.38, 2, 100)
check("... bootstrap lower bound (pp)", np.percentile(boot, 2.5), -11.32, 2, 100)
check("... bootstrap upper bound (pp)", np.percentile(boot, 97.5), -1.68, 2, 100)

# Unknown-indication bounds
k_m, n_m, k_c, n_c = (
    primary_test[x] for x in ("k_medical", "n_medical", "k_cosmetic", "n_cosmetic")
)
k_u, n_u = unknown_reporting, len(unknown)
to_medical = (k_m + k_u) / (n_m + n_u) - k_c / n_c
to_cosmetic = k_m / n_m - (k_c + k_u) / (n_c + n_u)
check("All unknown assigned medical: difference (pp)", to_medical, 0.24, 2, 100)
check("... lower bound (pp)", newcombe(k_m + k_u, n_m + n_u, k_c, n_c)[0], -0.77, 2, 100)
check("All unknown assigned cosmetic: difference (pp)", to_cosmetic, -6.26, 2, 100)
check("... upper bound (pp)", newcombe(k_m, n_m, k_c + k_u, n_c + n_u)[1], -5.31, 2, 100)

### 7.9 Analyses added after the October 2026 review

A friendly review of the draft asked for several further numbers. Most come from data this notebook
has already loaded; they are computed by `13_review_analyses.py`, which this section imports, as
section 7.6 imports `08_robustness_analyses.py`. The script only reads private inputs and writes
aggregates.

In [ ]:
# Load the review-analysis script as a module, as section 7.6 does for 08
spec = importlib.util.spec_from_file_location("review_analyses", ANALYSIS / "13_review_analyses.py")
review = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = review
spec.loader.exec_module(review)

verified = review.verified_outputs()  # per record and model: output values and majority verdicts
truth = review.truth_tables(verified)  # the true values that the binary verdicts determine

**What.** The sensitivity and positive predictive value of each verified model's self-use decision,
and the accuracy of two runs that were never annotated, the deployed batch run and DeepSeek with high
reasoning, scored against the same human verdicts.

**Why.** Self-use gates every other field, so its errors in both directions matter, not only its
accuracy. Because self-use and side-effect presence are binary, a verdict on one model's value also
fixes the true value: if the model said *true* and was judged wrong, the truth is *false*. That makes
it possible to score the deployed run without new annotation. For the indication, what matters for the
comparison is whether a medical or cosmetic label is right when one is given.

In [ ]:
gate = review.self_use_gate(verified)
check("GPT-5.6 Luna self-use sensitivity (%)", gate["stage3"]["sensitivity"]["rate"], 98.7, 1, 100)
check(
    "GPT-5.6 Luna self-use positive predictive value (%)",
    gate["stage3"]["ppv"]["rate"],
    99.0,
    1,
    100,
)
check("DeepSeek self-use sensitivity (%)", gate["stage1"]["sensitivity"]["rate"], 86.1, 1, 100)
check(
    "DeepSeek self-use positive predictive value (%)", gate["stage1"]["ppv"]["rate"], 93.3, 1, 100
)

runs = review.deployed_runs(truth)
check(
    "Batch run: self-use accuracy (%)", runs["stage4"]["self_use"]["accuracy"]["rate"], 98.3, 1, 100
)
check("Batch run: medical labels right", runs["stage4"]["medical_label_precision"]["k"], 201)
check("... of medical labels given", runs["stage4"]["medical_label_precision"]["n"], 205)
check("Batch run: cosmetic labels right", runs["stage4"]["cosmetic_label_precision"]["k"], 110)
check("... of cosmetic labels given", runs["stage4"]["cosmetic_label_precision"]["n"], 112)
check(
    "DeepSeek, high reasoning: self-use accuracy (%)",
    runs["stage2"]["self_use"]["accuracy"]["rate"],
    93.3,
    1,
    100,
)

**What.** For each optional field, how often an empty value was judged wrong among GPT-5.6 Luna's correct
self-use outputs.

**Why.** The conditional accuracies in Table 1 measure precision where a field holds a value. They cannot
show whether the low disclosure rates in section 4 reflect what posts say or what the model missed; an
empty value judged wrong is exactly a miss.

In [ ]:
misses = review.sparse_field_misses(verified)["stage3"]
display(pd.DataFrame(misses).T[["label", "k", "n", "rate"]])
others = [m["rate"] for f, m in misses.items() if f != "c_indication_subtype" and m["n"]]
check("Highest miss rate outside the indication subtype (%)", max(others), 6.8, 1, 100)
check("Indication subtype: empty values judged wrong", misses["c_indication_subtype"]["k"], 28)
check("... of empty subtypes", misses["c_indication_subtype"]["n"], 67)

**What.** Coverage of the 118,762 submissions sent for extraction (Table 4), the submissions whose author
cannot be linked, the 1% sample before the content filter, and the retry history of the batch run.

**Why.** Table 4 counted coverage over all 121,019 submissions with text, which includes the *bonta*-only
false positives; the extraction set is the population the analysis uses. Unlinkable authors are kept as
single observations in every author-level analysis, so their number bounds how much that choice can
matter.

In [ ]:
corpus_tables = review.corpus_tables()
review_records = corpus_tables.pop("records")
eligible = corpus_tables["coverage"]["eligible"]
check("Extraction set: subreddits", eligible["communities"], 13106)
check("Extraction set: named authors", eligible["named_authors"], 79545)
check("Extraction set: median body length (characters)", eligible["median_body_characters"], 637)
a = corpus_tables["authors"]
unlinked = sum(n for key, n in a.items() if not key.endswith("|all"))
check("Unlinkable submissions", unlinked, 867)
check("... by deleted accounts", sum(n for key, n in a.items() if key.endswith("|[deleted]")), 848)
check("... by AutoModerator", sum(n for key, n in a.items() if key.endswith("|AutoModerator")), 19)
check(
    "Unlinkable medical-use submissions",
    a.get("medical|[deleted]", 0) + a.get("medical|AutoModerator", 0),
    99,
)
check(
    "Unlinkable cosmetic-use submissions",
    a.get("cosmetic|[deleted]", 0) + a.get("cosmetic|AutoModerator", 0),
    58,
)
check("AutoModerator cosmetic self-use submissions", a.get("cosmetic|AutoModerator", 0), 2)

con4 = sqlite3.connect(f"file:{PRIVATE / 'private_extraction.sqlite3'}?mode=ro", uri=True)
stage3_failed = con4.execute(
    "SELECT COUNT(DISTINCT record_id) FROM stage_failures WHERE stage = 'stage3'"
).fetchone()[0]
check("1% sample drawn", len(views["stage3"]) + stage3_failed, 1187)
check("... blocked by the content filter", stage3_failed, 19)
flagged = con4.execute(
    "SELECT COUNT(DISTINCT record_id) FROM stage_failure_history WHERE stage = 'stage4'"
).fetchone()[0]
terminal = con4.execute(
    "SELECT COUNT(DISTINCT record_id) FROM stage_failures WHERE stage = 'stage4'"
).fetchone()[0]
invalid = con4.execute(
    "SELECT COUNT(*) FROM stage_failure_history WHERE stage = 'stage4' AND error LIKE 'ValidationError%'"
).fetchone()[0]
check("Batch run: flagged and then succeeded on retry", flagged - terminal, 23)
check("Batch run: responses failing validation", invalid, 0)
con4.close()

**What.** Where the 5,856 excluded unknown-indication submissions come from, and two reallocations of
them that use the data rather than the extreme bounds of section 7.8: (a) those posted in r/noburp go to
the medical group; (b) every one goes to the medical group, the cosmetic group or stays excluded, with the
probabilities seen among the 64 verified unknown-indication outputs, separately for r/noburp and other
communities. A verified output's true indication is taken from the other model where that was judged
right. Each of 2,000 replicates resamples the verified outputs first.

**Why.** The excluded group reports side effects more often than either analysed group, and half of its
verified labels were wrong, so where it belongs can move the prevalence difference. The extreme bounds
assume all of it belongs to one group, which the verification shows is not the case.

In [ ]:
unknown_review = review.unknown_stratum(
    review_records, verified, np.random.default_rng(review.SEED)
)
u = unknown_review
check("Unknown-indication submissions in r/noburp", u["noburp"]["submissions"], 1775)
check(
    "... with a candidate side effect (%)",
    u["noburp"]["with_side_effect"] / u["noburp"]["submissions"],
    78.3,
    1,
    100,
)
check(
    "Unknown indication outside r/noburp: side effect (%)",
    u["outside_noburp"]["rate"],
    33.0,
    1,
    100,
)
split = u["verified_split"]
check(
    "Wrong 'unknown' recovered as medical",
    split["noburp"].get("medical", 0) + split["other"].get("medical", 0),
    15,
)
check("... of them in r/noburp", split["noburp"].get("medical", 0), 8)
check("Wrong 'unknown' recovered as cosmetic", split["other"].get("cosmetic", 0), 14)
c = u["noburp_to_medical"]
check("r/noburp unknowns to medical: difference (pp)", c["difference"], 0.78, 2, 100)
check("... lower bound (pp)", c["ci_low"], -0.27, 2, 100)
check("... upper bound (pp)", c["ci_high"], 1.82, 2, 100)
check(
    "... Pearson P",
    stats.chi2_contingency(
        [
            [c["medical_k"], c["medical_n"] - c["medical_k"]],
            [c["cosmetic_k"], c["cosmetic_n"] - c["cosmetic_k"]],
        ],
        correction=False,
    )[1],
    0.14,
    2,
)
r = u["verified_reallocation"]
check("Verified reallocation: median difference (pp)", r["median"], -1.14, 2, 100)
check("... 2.5th percentile (pp)", r["p2_5"], -1.88, 2, 100)
check("... 97.5th percentile (pp)", r["p97_5"], -0.37, 2, 100)
noburp = review.noburp_without_eructation(review_records)
check(
    "r/noburp medical use: side effect without eructation (%)",
    noburp["without_eructation"]["rate"],
    68.3,
    1,
    100,
)

**What.** The yield of Fig. 2, the attitude distribution and the distinct-terms test after merging synonyms
and removing terms that restate the treated condition or the intended effect (the vocabulary of section
7.6); the period-specific prevalence differences; and the side-effect rates behind the two within-region
differences of section 7.5.

**Why.** Terms such as eructation in r/noburp are not adverse events, and a submission whose only term is
one of them counts as a side-effect report. Recomputing without them shows how much of the yield, and of
the attitude split, they carry. The within-region differences are large, so the rates behind them belong
in the text.

In [ ]:
vy = review.vocabulary_yield(review_records)
check("Cleaned vocabulary: submissions with a side effect", vy["any_cleaned"], 13165)
check("Cleaned vocabulary: submissions with more than one", vy["multiple_cleaned"], 7276)
for attitude, value in (("distressed", 84.1), ("accepting", 15.2), ("dismissive", 0.7)):
    check(
        f"Cleaned vocabulary: {attitude} (%)",
        vy["attitude_cleaned"][attitude]["rate"],
        value,
        1,
        100,
    )
dt = review.distinct_terms_cleaned(review_records)
check("Cleaned vocabulary: medical mean distinct terms", dt["medical_mean"], 2.33, 2)
check("Cleaned vocabulary: cosmetic mean distinct terms", dt["cosmetic_mean"], 2.08, 2)
check("Cleaned vocabulary: Mann-Whitney P (x 1e13)", dt["p"] * 1e13, 1.3, 1)
check("Cleaned vocabulary: rank-biserial correlation", dt["rank_biserial"], 0.080, 3)
check("Cleaned medical denominator", dt["medical_n"], 6302)
check("Cleaned cosmetic denominator", dt["cosmetic_n"], 4284)
check("Medical rate factor", primary_test["k_medical"] / dt["medical_n"], 1.09, 2)
check("Cosmetic rate factor", primary_test["k_cosmetic"] / dt["cosmetic_n"], 1.06, 2)

for period, value in (("2015-2019", -0.87), ("2020-2022", -2.82), ("2023-2025", -2.37)):
    check(f"{period}: difference (pp)", period_table.loc[period, "difference pp"], value, 2)

for name, values in (
    ("chronic migraine vs forehead or glabellar lines", (21.6, 43.7, 6857, 3956)),
    ("temporomandibular disorder vs masseter or jawline", (32.3, 43.9, 3344, 1593)),
):
    medical_subtypes, cosmetic_subtypes = PAIRS[name]
    med = [r for r in recs if r.arm == "medical" and r.subtype in medical_subtypes]
    cos = [r for r in recs if r.arm == "cosmetic" and r.subtype in cosmetic_subtypes]
    check(
        f"{name}: medical side effect (%)",
        sum(bool(r.terms) for r in med) / len(med),
        values[0],
        1,
        100,
    )
    check(
        f"{name}: cosmetic side effect (%)",
        sum(bool(r.terms) for r in cos) / len(cos),
        values[1],
        1,
        100,
    )
    check(f"{name}: medical n", len(med), values[2])
    check(f"{name}: cosmetic n", len(cos), values[3])

**What.** (i) The bootstrap mean and basic interval of the divergence, from the author-clustered bootstrap of
section 7.2 rerun with its seed. (ii) The range of each group's corrected prevalence across the replicates
of section 7.8. (iii) The two rows added to Table 3.

**Why.** (i) The plug-in divergence is biased upward under resampling, so its percentile interval sits
above the estimate; the basic interval reflects it back. (ii) A corrected prevalence could in principle
leave the range zero to one, which would need truncation. (iii) The review asked for the next genuine
adverse events after migraine and eructation, which are not adverse events.

In [ ]:
jb = review.jsd_bootstrap()
check("Divergence: bootstrap mean", jb["jsd_bootstrap_mean"], 0.382, 3)
check("Divergence: basic interval, lower", jb["jsd_basic_ci_low"], 0.342, 3)
check("Divergence: basic interval, upper", jb["jsd_basic_ci_high"], 0.381, 3)

replay = np.random.default_rng(RESAMPLING_SEED)  # the generator of section 7.8, replayed
ranges = {arm: [] for arm in ("medical", "cosmetic")}
for _ in range(2_000):
    resampled = {
        arm: [
            luna_outputs[arm][i]
            for i in replay.integers(0, len(luna_outputs[arm]), len(luna_outputs[arm]))
        ]
        for arm in ("medical", "cosmetic")
    }
    try:
        values = {arm: observed[arm] * correction_factor(resampled[arm]) for arm in ranges}
    except ZeroDivisionError:
        continue
    for arm in ranges:
        ranges[arm].append(values[arm])
check("Corrected medical prevalence: lowest replicate (%)", min(ranges["medical"]), 26.3, 1, 100)
check("Corrected medical prevalence: highest replicate (%)", max(ranges["medical"]), 34.4, 1, 100)
check("Corrected cosmetic prevalence: lowest replicate (%)", min(ranges["cosmetic"]), 30.3, 1, 100)
check("Corrected cosmetic prevalence: highest replicate (%)", max(ranges["cosmetic"]), 44.6, 1, 100)

table3 = {row["term"]: row for row in review.term_table()}
for term, (diff, low, high) in (
    ("abdominal distension", (3.64, 3.13, 4.19)),
    ("sore throat", (3.53, 3.08, 4.00)),
):
    check(f"Table 3, {term}: difference", table3[term]["difference"], diff, 2)
    check("... clustered lower bound", table3[term]["clustered_ci_low"], low, 2)
    check("... clustered upper bound", table3[term]["clustered_ci_high"], high, 2)

**What.** The post-level presence labels for the nine GPT-5.6 Luna outputs counted as side-effect false
positives in the compared groups, which the three annotators gave independently with a labelling form
(majority vote), and an author's reading of the three unknown-indication outputs neither model got right.
The labels are in a private file next to the extraction records, because it names records.

**Why.** The presence rule counts a non-empty list with no correct span as a false positive even when the
post describes some other side effect, so whether these posts describe a side effect had to be judged
separately. By majority vote none does, so the post-level counts equal the strict rule's and the
correction of section 7.8 stands (an author's earlier reading had counted two). The cell is skipped when
the private file is absent.

In [ ]:
adjudicated = review.adjudication(review_records, verified)
if adjudicated is None:
    print("No private adjudication file; skipped.")
else:
    check(
        "Re-labelled false positives judged to describe a side effect",
        sum(adjudicated["reclassified_as_true_positive"].values()),
        0,
    )
    check(
        "Medical positive predictive value, post level (%)",
        adjudicated["ppv"]["medical"],
        90.4,
        1,
        100,
    )
    corr = adjudicated["correction"]
    check("Corrected difference, post level (pp)", corr["corrected_difference"], -6.38, 2, 100)
    check("... bootstrap lower bound (pp)", corr["corrected_difference_ci_low"], -11.32, 2, 100)
    check("... bootstrap upper bound (pp)", corr["corrected_difference_ci_high"], -1.68, 2, 100)
    check(
        "Verified reallocation after re-reading: median (pp)",
        adjudicated["unknown_verified_reallocation"]["median"],
        -1.15,
        2,
        100,
    )
    for arm, (s, s_lo, s_hi, ppv, p_lo, p_hi) in (
        ("medical", (98.5, 92.0, 99.7, 90.4, 81.5, 95.3)),
        ("cosmetic", (92.7, 80.6, 97.5, 95.0, 83.5, 98.6)),
    ):
        check(
            f"Post-level presence, {arm}: sensitivity (%)",
            adjudicated["sensitivity"][arm],
            s,
            1,
            100,
        )
        check("... lower bound (%)", adjudicated["sensitivity_ci"][arm][0], s_lo, 1, 100)
        check("... upper bound (%)", adjudicated["sensitivity_ci"][arm][1], s_hi, 1, 100)
        check(
            f"Post-level presence, {arm}: positive predictive value (%)",
            adjudicated["ppv"][arm],
            ppv,
            1,
            100,
        )
        check("... lower bound (%)", adjudicated["ppv_ci"][arm][0], p_lo, 1, 100)
        check("... upper bound (%)", adjudicated["ppv_ci"][arm][1], p_hi, 1, 100)
    span = adjudicated["corrected_prevalence_range"]
    check(
        "Post-level corrected medical prevalence: lowest replicate (%)",
        span["medical"][0],
        26.3,
        1,
        100,
    )
    check(
        "Post-level corrected medical prevalence: highest replicate (%)",
        span["medical"][1],
        34.4,
        1,
        100,
    )
    check(
        "Post-level corrected cosmetic prevalence: lowest replicate (%)",
        span["cosmetic"][0],
        30.3,
        1,
        100,
    )
    check(
        "Post-level corrected cosmetic prevalence: highest replicate (%)",
        span["cosmetic"][1],
        44.6,
        1,
        100,
    )

**What.** How often the deployed batch run returned the same candidate-term set as the verified
interactive run, and the full-corpus row of Supplementary Table 2 (DeepSeek against the batch run over all
118,747 shared records).

**Why.** The human verification describes the interactive run. Both runs are the same model at the same
settings with provider-default sampling and no seed, so differences between them are sampling variation;
where the term sets agree, the verdicts apply to the batch run unchanged. Supplementary Table 2 had shown
only the nested 10% sample for the full-corpus comparison that Supplementary Note 2 describes.

In [ ]:
identity = review.batch_list_identity(verified)
check("Batch run: same candidate-term set as the verified run", identity["same_term_set"], 237)
check("... of verified medical and cosmetic self-use submissions", identity["both_self_use"], 319)

full = review.full_corpus_agreement()
check("Full corpus, DeepSeek vs batch: paired records", full["paired_records"], 118747)
check("... self-use kappa", full["self_use_kappa"], 0.839, 3)
check("... side-effect presence kappa", full["presence_kappa"], 0.817, 3)
check("... exact candidate-term set", full["term_set_exact"], 0.638, 3)
check("... mean Jaccard", full["term_set_jaccard"], 0.689, 3)
check("... full-record agreement", full["full_record"], 0.383, 3)

## 8. Figures

**What.** Figures 1 to 4 of the manuscript, redrawn at print size by the figure script
`11_manuscript_figures.py`, into this notebook's output folder.

**Why.** The figure script reads the archived aggregate outputs of the analysis scripts. Sections 1 to 7
have recomputed every number those outputs contain and checked them against the paper, so drawing from
the archived aggregates shows exactly what was printed. The figures are written to
`12_notebook_outputs/figures/` and the manuscript's own figure folder is left untouched.

In [ ]:
from IPython.display import Image

spec = importlib.util.spec_from_file_location(
    "manuscript_figures", ANALYSIS / "11_manuscript_figures.py"
)
figures = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = figures
spec.loader.exec_module(figures)
figures.FIGURES_DIR = OUT / "figures"  # write here instead of data_analysis/figures
figures.MANUSCRIPT_FIGURES = (
    OUT / "_do_not_copy"
)  # a folder that does not exist: no copy into the paper
figures.style()
figures.figure_study_overview()  # Figure 1
figures.figure_overview()
figures.figure_profile()
figures.figure_robustness()
for name in (
    "11_fig0_study_overview",
    "11_fig1_extraction_overview",
    "11_fig2_term_profile",
    "11_fig3_robustness",
):
    display(Image(filename=str(OUT / "figures" / f"{name}.png"), width=720))

## 9. Reproduction report

**What.** Every check made above, with the value printed in the paper, the recomputed value and the
outcome. The table is also written to `12_notebook_outputs/reproduction_checks.csv`.

**Why.** One table makes it easy to see whether anything failed to reproduce, and gives a record to
archive with the code.

In [ ]:
report = pd.DataFrame(CHECKS)
report.to_csv(OUT / "reproduction_checks.csv", index=False)
print(f"{(report['result'] == 'PASS').sum()} of {len(report)} checks passed")
failed = report[report["result"] == "FAIL"]
display(failed if len(failed) else report.tail(5))
con.close()